In [1]:
import pyspark
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os


# Business Exploration

In [3]:
import pandas as pd

business_df = pd.read_json(
    "/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/YelpDataset/yelp_academic_dataset_business.json",
    lines=True
)

business_df.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."


In [4]:
business_df.shape

(150346, 14)

In [5]:
business_df.columns

Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'attributes', 'categories', 'hours'],
      dtype='object')

In [6]:
business_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   business_id   150346 non-null  object 
 1   name          150346 non-null  object 
 2   address       150346 non-null  object 
 3   city          150346 non-null  object 
 4   state         150346 non-null  object 
 5   postal_code   150346 non-null  object 
 6   latitude      150346 non-null  float64
 7   longitude     150346 non-null  float64
 8   stars         150346 non-null  float64
 9   review_count  150346 non-null  int64  
 10  is_open       150346 non-null  int64  
 11  attributes    136602 non-null  object 
 12  categories    150243 non-null  object 
 13  hours         127123 non-null  object 
dtypes: float64(3), int64(2), object(9)
memory usage: 16.1+ MB


## Check data quality

In [7]:
business_df.isnull().sum()

business_id         0
name                0
address             0
city                0
state               0
postal_code         0
latitude            0
longitude           0
stars               0
review_count        0
is_open             0
attributes      13744
categories        103
hours           23223
dtype: int64

In [8]:
# Missing values
missing_summary = (
    business_df.isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(business_df) * 100
).round(2)

missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

,missing_count,missing_percentage
hours,23223,15.45
attributes,13744,9.14
categories,103,0.07
business_id,0,0.00
name,0,0.00
address,0,0.00
city,0,0.00
state,0,0.00
postal_code,0,0.00
latitude,0,0.00


In [9]:
text_columns = [
    "business_id",
    "name",
    "address",
    "city",
    "state",
    "postal_code",
    "categories"
]

blank_summary = pd.DataFrame({
    "blank_count": [
        business_df[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
        for column in text_columns
    ]
}, index=text_columns)

blank_summary["blank_percentage"] = (
    blank_summary["blank_count"] / len(business_df) * 100
).round(2)

blank_summary

,blank_count,blank_percentage
business_id,0,0.00
name,0,0.00
address,5127,3.41
city,0,0.00
state,0,0.00
postal_code,73,0.05
categories,0,0.00


In [10]:
business_df.assign(
    attributes=business_df["attributes"].astype(str),
    hours=business_df["hours"].astype(str)
).duplicated().sum()

0

In [11]:
# Duplicate identifiers
print(
    "Duplicate business IDs:",
    business_df["business_id"].duplicated().sum()
)

Duplicate business IDs: 0


In [12]:
# Numerical summary
business_df[
    ["stars", "review_count", "latitude", "longitude"]
].describe()

,stars,review_count,latitude,longitude
count,150346.000000,150346.000000,150346.000000,150346.000000
mean,3.596724,44.866561,36.671150,-89.357339
std,0.974421,121.120136,5.872759,14.918502
min,1.000000,5.000000,27.555127,-120.095137
25%,3.000000,8.000000,32.187293,-90.357810
50%,3.500000,15.000000,38.777413,-86.121179
75%,4.500000,37.000000,39.954036,-75.421542
max,5.000000,7568.000000,53.679197,-73.200457


In [13]:
print(
    "Invalid star ratings:",
    (~business_df["stars"].between(1, 5)).sum()
)

print(
    "Negative review counts:",
    (business_df["review_count"] < 0).sum()
)

Invalid star ratings: 0
Negative review counts: 0


In [14]:
business_df["is_open"].value_counts(dropna=False)

is_open
1    119698
0     30648
Name: count, dtype: int64

In [15]:
business_df["city_clean"] = (
    business_df["city"]
    .fillna("")
    .str.strip()
    .str.title()
)

business_df["state_clean"] = (
    business_df["state"]
    .fillna("")
    .str.strip()
    .str.upper()
)

In [16]:
location_summary = (
    business_df
    .groupby(["state_clean", "city_clean"], as_index=False)
    .agg(
        business_count=("business_id", "nunique"),
        total_reviews=("review_count", "sum"),
        median_reviews=("review_count", "median"),
        open_businesses=("is_open", "sum")
    )
    .sort_values(
        ["business_count", "total_reviews"],
        ascending=False
    )
)

location_summary.head(30)

,state_clean,city_clean,business_count,total_reviews,median_reviews,open_businesses
1088,PA,Philadelphia,14575,936558,19.0,10547
45,AZ,Tucson,9261,387647,15.0,7544
259,FL,Tampa,9067,440062,15.0,7235
393,IN,Indianapolis,7545,349322,16.0,5899
1257,TN,Nashville,6978,441203,18.0,5405
456,LA,New Orleans,6215,621518,22.0,4654
812,NV,Reno,5937,334766,18.0,4764
6,AB,Edmonton,5056,98274,10.0,3918
537,MO,Saint Louis,4832,244422,15.0,3408
70,CA,Santa Barbara,3836,262961,18.0,3027


In [17]:
business_df["category_list"] = (
    business_df["categories"]
    .fillna("")
    .str.split(", ")
)

category_counts = (
    business_df[["business_id", "category_list"]]
    .explode("category_list")
    .query("category_list != ''")
    .groupby("category_list")["business_id"]
    .nunique()
    .sort_values(ascending=False)
)

category_counts.head(40)

category_list
Restaurants                  52268
Food                         27781
Shopping                     24395
Home Services                14356
Beauty & Spas                14292
Nightlife                    12281
Health & Medical             11890
Local Services               11198
Bars                         11065
Automotive                   10773
Event Planning & Services     9895
Sandwiches                    8366
American (Traditional)        8139
Active Life                   7687
Pizza                         7093
Coffee & Tea                  6703
Fast Food                     6472
Breakfast & Brunch            6239
American (New)                6097
Hotels & Travel               5857
Home & Garden                 5799
Fashion                       5739
Burgers                       5636
Arts & Entertainment          5434
Auto Repair                   5433
Hair Salons                   5046
Nail Salons                   4621
Mexican                       4600
Italia

In [18]:
missing_categories = (
    business_df["categories"].isna()
    | business_df["categories"].str.strip().eq("")
)

print(
    "Businesses without categories:",
    missing_categories.sum()
)

Businesses without categories: 103


In [19]:
business_audit = pd.DataFrame({
    "metric": [
        "Total businesses",
        "Unique business IDs",
        "Open businesses",
        "Businesses with categories",
        "Businesses with attributes",
        "Businesses with operating hours",
        "Distinct cities",
        "Distinct states/regions"
    ],
    "value": [
        len(business_df),
        business_df["business_id"].nunique(),
        business_df["is_open"].eq(1).sum(),
        business_df["categories"].notna().sum(),
        business_df["attributes"].notna().sum(),
        business_df["hours"].notna().sum(),
        business_df["city"].nunique(),
        business_df["state"].nunique()
    ]
})

business_audit

,metric,value
0,Total businesses,150346
1,Unique business IDs,150346
2,Open businesses,119698
3,Businesses with categories,150243
4,Businesses with attributes,136602
5,Businesses with operating hours,127123
6,Distinct cities,1416
7,Distinct states/regions,27


# Image Exploration

In [20]:
photos_df = pd.read_json("/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/Image_files/photos.json", lines= True)

photos_df.head()

,photo_id,business_id,caption,label
0,zsvj7vloL4L5jhYyPIuVwg,Nk-SJhPlDBkAZvfsADtccA,Nice rock artwork everywhere and craploads of ...,inside
1,HCUdRJHHm_e0OCTlZetGLg,yVZtL5MmrpiivyCIrVkGgA,,outside
2,vkr8T0scuJmGVvN2HJelEA,_ab50qdWOk0DdB6XOrBitw,oyster shooter,drink
3,pve7D6NUrafHW3EAORubyw,SZU9c8V2GuREDN5KgyHFJw,Shrimp scampi,food
4,H52Er-uBg6rNrHcReWTD2w,Gzur0f0XMkrVxIwYJvOt2g,,food


In [21]:
photos_df.isna().sum()

photo_id       0
business_id    0
caption        0
label          0
dtype: int64

In [22]:
photos_df.columns

Index(['photo_id', 'business_id', 'caption', 'label'], dtype='object')

In [23]:
photos_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200100 entries, 0 to 200099
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   photo_id     200100 non-null  object
 1   business_id  200100 non-null  object
 2   caption      200100 non-null  object
 3   label        200100 non-null  object
dtypes: object(4)
memory usage: 6.1+ MB


## Check duplicates, blank captions and labels

In [24]:
photo_audit = pd.Series({
    "total_photo_records": len(photos_df),
    "unique_photo_ids": photos_df["photo_id"].nunique(),
    "duplicate_photo_ids": photos_df["photo_id"].duplicated().sum(),
    "businesses_with_photos": photos_df["business_id"].nunique(),
    "blank_captions": (
        photos_df["caption"]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    ),
    "unique_labels": photos_df["label"].nunique()
})

photo_audit

total_photo_records       200100
unique_photo_ids          200098
duplicate_photo_ids            2
businesses_with_photos     36680
blank_captions            103366
unique_labels                  5
dtype: int64

In [25]:
photo_label_summary = (
    photos_df["label"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="photo_count")
)

photo_label_summary["percentage"] = (
    photo_label_summary["photo_count"]
    / len(photos_df)
    * 100
).round(2)

photo_label_summary

,label,photo_count,percentage
0,food,108152,54.05
1,inside,56031,28.00
2,outside,18569,9.28
3,drink,15670,7.83
4,menu,1678,0.84


In [26]:
# Ensure the cleaned location columns exist
if "state_clean" not in business_df.columns:
    business_df["state_clean"] = (
        business_df["state"]
        .fillna("")
        .str.strip()
        .str.upper()
    )

if "city_clean" not in business_df.columns:
    business_df["city_clean"] = (
        business_df["city"]
        .fillna("")
        .str.strip()
        .str.title()
    )

# Create the provisional New Orleans business subset
new_orleans_businesses = (
    business_df.loc[
        (business_df["state_clean"] == "LA")
        & (business_df["city_clean"] == "New Orleans")
        & (business_df["is_open"] == 1)
        & (business_df["categories"].notna())
    ]
    .copy()
    .reset_index(drop=True)
)

print("New Orleans businesses:", len(new_orleans_businesses))
print(
    "Total recorded reviews:",
    new_orleans_businesses["review_count"].sum()
)

New Orleans businesses: 4653
Total recorded reviews: 525131


In [27]:
# Inspect the duplicate photo IDs
duplicate_photo_rows = photos_df.loc[
    photos_df["photo_id"].duplicated(keep=False)
].sort_values("photo_id")

duplicate_photo_rows

,photo_id,business_id,caption,label
29695,_CYoxbCIKuAwpq4crHCPWg,B-NqcrIhvzCIkLCCpmhdvg,,inside
174504,_CYoxbCIKuAwpq4crHCPWg,B-NqcrIhvzCIkLCCpmhdvg,,drink
173977,qtl5YDIc2q0yelW_e9FfqQ,fBCv5Euudl9VieR870gwNg,,inside
188850,qtl5YDIc2q0yelW_e9FfqQ,fBCv5Euudl9VieR870gwNg,,food


In [28]:
# Retain one record per photo ID
photos_df_clean = (
    photos_df
    .drop_duplicates(subset="photo_id", keep="first")
    .copy()
)

print("Photos after duplicate removal:", len(photos_df_clean))

Photos after duplicate removal: 200098


In [29]:
# Filter photos to the New Orleans businesses
new_orleans_business_ids = set(
    new_orleans_businesses["business_id"]
)

new_orleans_photos = (
    photos_df_clean.loc[
        photos_df_clean["business_id"].isin(
            new_orleans_business_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print("New Orleans photos:", len(new_orleans_photos))
print(
    "Businesses with at least one photo:",
    new_orleans_photos["business_id"].nunique()
)

New Orleans photos: 16847
Businesses with at least one photo: 1425


In [30]:
business_photo_counts = (
    new_orleans_photos
    .groupby("business_id")
    .size()
    .rename("photo_count")
)

new_orleans_photo_coverage = (
    new_orleans_businesses[
        ["business_id", "name", "categories", "review_count"]
    ]
    .merge(
        business_photo_counts,
        on="business_id",
        how="left"
    )
)

new_orleans_photo_coverage["photo_count"] = (
    new_orleans_photo_coverage["photo_count"]
    .fillna(0)
    .astype(int)
)

new_orleans_photo_coverage["has_photo"] = (
    new_orleans_photo_coverage["photo_count"] > 0
)

coverage_percentage = (
    new_orleans_photo_coverage["has_photo"].mean() * 100
)

print(
    "Photo coverage:",
    round(coverage_percentage, 2),
    "%"
)

new_orleans_photo_coverage["photo_count"].describe()

Photo coverage: 30.63 %


count    4653.000000
mean        3.620675
std        15.290308
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max       528.000000
Name: photo_count, dtype: float64

In [31]:
#check the useful coverage thresholds

photo_coverage_levels = pd.Series({
    "Total businesses": len(new_orleans_photo_coverage),

    "No photos": (
        new_orleans_photo_coverage["photo_count"] == 0
    ).sum(),

    "At least 1 photo": (
        new_orleans_photo_coverage["photo_count"] >= 1
    ).sum(),

    "At least 3 photos": (
        new_orleans_photo_coverage["photo_count"] >= 3
    ).sum(),

    "At least 5 photos": (
        new_orleans_photo_coverage["photo_count"] >= 5
    ).sum()
})

photo_coverage_levels

Total businesses     4653
No photos            3228
At least 1 photo     1425
At least 3 photos     991
At least 5 photos     714
dtype: int64

In [32]:
# calculate the business-level image coverage

business_photo_counts = (
    new_orleans_photos
    .groupby("business_id")
    .size()
    .rename("photo_count")
)

new_orleans_photo_coverage = (
    new_orleans_businesses[
        ["business_id", "name", "categories", "review_count"]
    ]
    .merge(
        business_photo_counts,
        on="business_id",
        how="left"
    )
)

new_orleans_photo_coverage["photo_count"] = (
    new_orleans_photo_coverage["photo_count"]
    .fillna(0)
    .astype(int)
)

new_orleans_photo_coverage["has_photo"] = (
    new_orleans_photo_coverage["photo_count"] > 0
)

coverage_percentage = (
    new_orleans_photo_coverage["has_photo"].mean() * 100
)

print(
    "Photo coverage:",
    round(coverage_percentage, 2),
    "%"
)

new_orleans_photo_coverage["photo_count"].describe()

Photo coverage: 30.63 %


count    4653.000000
mean        3.620675
std        15.290308
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max       528.000000
Name: photo_count, dtype: float64

In [33]:
#Inspect the New Orleans image-label distribution
new_orleans_label_summary = (
    new_orleans_photos["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="photo_count")
)

new_orleans_label_summary["percentage"] = (
    new_orleans_label_summary["photo_count"]
    / len(new_orleans_photos)
    * 100
).round(2)

new_orleans_label_summary

,label,photo_count,percentage
0,food,8068,47.89
1,inside,5723,33.97
2,outside,1705,10.12
3,drink,1243,7.38
4,menu,108,0.64


# Business and Photos Explorations

In [34]:
import pandas as pd
import numpy as np

def parse_categories(value):
    """Return a clean list of Yelp categories."""
    if isinstance(value, list):
        return [
            str(category).strip()
            for category in value
            if str(category).strip()
        ]

    if pd.isna(value):
        return []

    return [
        category.strip()
        for category in str(value).split(",")
        if category.strip()
    ]


business_df["category_list"] = (
    business_df["categories"]
    .apply(parse_categories)
)

In [35]:
food_restaurant_categories = {
    "Restaurants",
    "Food",
    "Cafes",
    "Coffee & Tea",
    "Bakeries",
    "Desserts",
    "Specialty Food",
    "Food Delivery Services",
    "Caterers"
}

drink_nightlife_categories = {
    "Bars",
    "Nightlife",
    "Lounges",
    "Pubs",
    "Cocktail Bars",
    "Wine Bars",
    "Beer Bar",
    "Breweries",
    "Wineries",
    "Distilleries"
}

hospitality_categories = {
    "Hotels",
    "Bed & Breakfast",
    "Resorts",
    "Venues & Event Spaces"
}

target_categories = (
    food_restaurant_categories
    | drink_nightlife_categories
    | hospitality_categories
)

In [36]:
business_df["matched_target_categories"] = (
    business_df["category_list"]
    .apply(
        lambda categories: sorted(
            set(categories).intersection(target_categories)
        )
    )
)

business_df["is_target_business"] = (
    business_df["matched_target_categories"]
    .apply(bool)
)

In [37]:
print(
    "Target businesses:",
    business_df["is_target_business"].sum()
)

print(
    "Target-business percentage:",
    round(
        business_df["is_target_business"].mean() * 100,
        2
    ),
    "%"
)

Target businesses: 71260
Target-business percentage: 47.4 %


In [38]:
business_df.loc[
    business_df["is_target_business"],
    [
        "name",
        "city",
        "categories",
        "matched_target_categories"
    ]
].head(20)

,name,city,categories,matched_target_categories
3,St Honore Pastries,Philadelphia,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...","[Bakeries, Coffee & Tea, Food, Restaurants]"
4,Perkiomen Valley Brewery,Green Lane,"Brewpubs, Breweries, Food","[Breweries, Food]"
5,Sonic Drive-In,Ashland City,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...","[Food, Restaurants]"
8,Tsevi's Pub And Grill,Affton,"Pubs, Restaurants, Italian, Bars, American (Tr...","[Bars, Nightlife, Pubs, Restaurants]"
9,Sonic Drive-In,Nashville,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,...","[Food, Restaurants]"
11,Vietnamese Food Truck,Tampa Bay,"Vietnamese, Food, Restaurants, Food Trucks","[Food, Restaurants]"
12,Denny's,Indianapolis,"American (Traditional), Restaurants, Diners, B...",[Restaurants]
14,Zio's Italian Market,Largo,"Food, Delis, Italian, Bakeries, Restaurants","[Bakeries, Food, Restaurants]"
15,Tuna Bar,Philadelphia,"Sushi Bars, Restaurants, Japanese",[Restaurants]
19,BAP,Philadelphia,"Korean, Restaurants",[Restaurants]


In [39]:
# Ensure cleaned location fields exist
business_df["state_clean"] = (
    business_df["state"]
    .fillna("")
    .str.strip()
    .str.upper()
)

business_df["city_clean"] = (
    business_df["city"]
    .fillna("")
    .str.strip()
    .str.title()
)

target_businesses = (
    business_df.loc[
        business_df["is_target_business"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("Target food and hospitality businesses:", len(target_businesses))

target_businesses["is_operational_at_snapshot"] = (
    target_businesses["is_open"].eq(1)
)

print("Open target businesses:", len(target_businesses))

Target food and hospitality businesses: 71260
Open target businesses: 71260


In [40]:
target_status_check = (
    business_df.loc[
        business_df["is_target_business"],
        "is_open"
    ]
    .value_counts(dropna=False)
    .rename_axis("is_open")
    .reset_index(name="business_count")
)

target_status_check

,is_open,business_count
0,1,49911
1,0,21349


In [41]:
all_target_businesses = (
    business_df.loc[
        business_df["is_target_business"]
    ]
    .copy()
    .reset_index(drop=True)
)

target_status_summary = pd.Series({
    "Total target businesses": len(all_target_businesses),
    "Open at dataset snapshot": (
        all_target_businesses["is_open"].eq(1).sum()
    ),
    "Closed at dataset snapshot": (
        all_target_businesses["is_open"].eq(0).sum()
    )
})

target_status_summary

Total target businesses       71260
Open at dataset snapshot      49911
Closed at dataset snapshot    21349
dtype: int64

In [42]:
photos_df_clean = (
    photos_df
    .drop_duplicates(
        subset="photo_id",
        keep="first"
    )
    .copy()
    .reset_index(drop=True)
)

photos_df_clean["caption_clean"] = (
    photos_df_clean["caption"]
    .fillna("")
    .astype(str)
    .str.strip()
)

photos_df_clean["has_caption"] = (
    photos_df_clean["caption_clean"].ne("")
)

In [43]:
photo_summary_by_business = (
    photos_df_clean
    .groupby("business_id", as_index=False)
    .agg(
        photo_count=("photo_id", "count"),
        captioned_photo_count=("has_caption", "sum"),
        unique_labels=("label", "nunique")
    )
)

In [44]:
business_photo_df = (
    target_businesses[
        [
            "business_id",
            "state_clean",
            "city_clean",
            "review_count",
            "is_operational_at_snapshot",
            "matched_target_categories"
        ]
    ]
    .merge(
        photo_summary_by_business,
        on="business_id",
        how="left"
    )
)

In [45]:
photo_columns = [
    "photo_count",
    "captioned_photo_count",
    "unique_labels"
]

business_photo_df[photo_columns] = (
    business_photo_df[photo_columns]
    .fillna(0)
    .astype(int)
)

business_photo_df["has_photo"] = (
    business_photo_df["photo_count"] > 0
)

business_photo_df["has_captioned_photo"] = (
    business_photo_df["captioned_photo_count"] > 0
)

In [46]:
photos_df_clean = (
    photos_df
    .drop_duplicates(
        subset="photo_id",
        keep="first"
    )
    .copy()
    .reset_index(drop=True)
)

photos_df_clean["caption_clean"] = (
    photos_df_clean["caption"]
    .fillna("")
    .astype(str)
    .str.strip()
)

photos_df_clean["has_caption"] = (
    photos_df_clean["caption_clean"].ne("")
)

In [47]:
photo_columns = [
    "photo_count",
    "captioned_photo_count",
    "unique_labels"
]

business_photo_df[photo_columns] = (
    business_photo_df[photo_columns]
    .fillna(0)
    .astype(int)
)

business_photo_df["has_photo"] = (
    business_photo_df["photo_count"] > 0
)

business_photo_df["has_captioned_photo"] = (
    business_photo_df["captioned_photo_count"] > 0
)

In [48]:
location_image_summary = (
    business_photo_df
    .groupby(
        ["state_clean", "city_clean"],
        as_index=False
    )
    .agg(
        business_count=("business_id", "nunique"),
        total_reviews=("review_count", "sum"),
        median_reviews=("review_count", "median"),

        businesses_with_photos=("has_photo", "sum"),
        businesses_with_captioned_photos=(
            "has_captioned_photo",
            "sum"
        ),

        total_photos=("photo_count", "sum"),
        captioned_photos=("captioned_photo_count", "sum"),

        mean_photos_per_business=("photo_count", "mean"),
        median_photos_per_business=("photo_count", "median"),
        p95_photos_per_business=(
            "photo_count",
            lambda values: values.quantile(0.95)
        ),
        maximum_photos_per_business=("photo_count", "max"),

        mean_unique_labels_per_business=(
            "unique_labels",
            "mean"
        )
    )
)

In [49]:
location_image_summary["photo_coverage_percentage"] = (
    location_image_summary["businesses_with_photos"]
    / location_image_summary["business_count"]
    * 100
)

location_image_summary[
    "captioned_business_coverage_percentage"
] = (
    location_image_summary[
        "businesses_with_captioned_photos"
    ]
    / location_image_summary["business_count"]
    * 100
)

location_image_summary["captioned_photo_percentage"] = (
    np.where(
        location_image_summary["total_photos"] > 0,
        (
            location_image_summary["captioned_photos"]
            / location_image_summary["total_photos"]
            * 100
        ),
        0
    )
)

In [50]:
percentage_columns = [
    "photo_coverage_percentage",
    "captioned_business_coverage_percentage",
    "captioned_photo_percentage",
    "mean_photos_per_business",
    "median_photos_per_business",
    "p95_photos_per_business",
    "mean_unique_labels_per_business"
]

location_image_summary[percentage_columns] = (
    location_image_summary[percentage_columns]
    .round(2)
)

In [51]:
top_five_location_images = (
    location_image_summary
    .sort_values(
        [
            "business_count",
            "businesses_with_photos"
        ],
        ascending=False
    )
    .head(5)
    .reset_index(drop=True)
)

top_five_location_images

,state_clean,city_clean,business_count,total_reviews,median_reviews,businesses_with_photos,businesses_with_captioned_photos,total_photos,captioned_photos,mean_photos_per_business,median_photos_per_business,p95_photos_per_business,maximum_photos_per_business,mean_unique_labels_per_business,photo_coverage_percentage,captioned_business_coverage_percentage,captioned_photo_percentage
0,PA,Philadelphia,7692,751438,31.0,4038,2904,25299,12730,3.29,1.0,14.0,201,0.94,52.50,37.75,50.32
1,FL,Tampa,4073,334036,30.0,2344,1726,15642,8218,3.84,1.0,17.0,191,1.09,57.55,42.38,52.54
2,IN,Indianapolis,3905,281357,28.0,2090,1425,11891,5746,3.05,1.0,14.0,157,0.98,53.52,36.49,48.32
3,TN,Nashville,3526,367155,34.0,1933,1421,14093,7088,4.00,1.0,19.0,171,1.08,54.82,40.30,50.29
4,LA,New Orleans,3505,547106,41.0,1977,1562,19456,10291,5.55,1.0,25.0,528,1.17,56.41,44.56,52.89


In [52]:
target_photo_records = (
    photos_df_clean
    .merge(
        target_businesses[
            [
                "business_id",
                "state_clean",
                "city_clean"
            ]
        ],
        on="business_id",
        how="inner"
    )
)

In [53]:
location_label_coverage = (
    target_photo_records
    .groupby(
        ["state_clean", "city_clean"],
        as_index=False
    )
    .agg(
        unique_photo_labels=("label", "nunique")
    )
)

In [54]:
top_five_location_images = (
    top_five_location_images
    .merge(
        location_label_coverage,
        on=["state_clean", "city_clean"],
        how="left"
    )
)

top_five_location_images

,state_clean,city_clean,business_count,total_reviews,median_reviews,businesses_with_photos,businesses_with_captioned_photos,total_photos,captioned_photos,mean_photos_per_business,median_photos_per_business,p95_photos_per_business,maximum_photos_per_business,mean_unique_labels_per_business,photo_coverage_percentage,captioned_business_coverage_percentage,captioned_photo_percentage,unique_photo_labels
0,PA,Philadelphia,7692,751438,31.0,4038,2904,25299,12730,3.29,1.0,14.0,201,0.94,52.50,37.75,50.32,5
1,FL,Tampa,4073,334036,30.0,2344,1726,15642,8218,3.84,1.0,17.0,191,1.09,57.55,42.38,52.54,5
2,IN,Indianapolis,3905,281357,28.0,2090,1425,11891,5746,3.05,1.0,14.0,157,0.98,53.52,36.49,48.32,5
3,TN,Nashville,3526,367155,34.0,1933,1421,14093,7088,4.00,1.0,19.0,171,1.08,54.82,40.30,50.29,5
4,LA,New Orleans,3505,547106,41.0,1977,1562,19456,10291,5.55,1.0,25.0,528,1.17,56.41,44.56,52.89,5


In [55]:
top_five_location_keys = (
    top_five_location_images[
        ["state_clean", "city_clean"]
    ]
)

top_five_photo_records = (
    target_photo_records
    .merge(
        top_five_location_keys,
        on=["state_clean", "city_clean"],
        how="inner"
    )
)

In [56]:
top_five_label_counts = (
    top_five_photo_records
    .groupby(
        [
            "state_clean",
            "city_clean",
            "label"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

top_five_label_counts

label,state_clean,city_clean,drink,food,inside,menu,outside
0,FL,Tampa,1358,8099,4487,133,1565
1,IN,Indianapolis,993,6269,3572,124,933
2,LA,New Orleans,1422,9321,6672,126,1915
3,PA,Philadelphia,1638,14033,7345,249,2034
4,TN,Nashville,1226,6293,4901,130,1543


In [57]:
label_columns = [
    column
    for column in top_five_label_counts.columns
    if column not in {
        "state_clean",
        "city_clean"
    }
]

top_five_label_percentages = (
    top_five_label_counts.copy()
)

label_totals = (
    top_five_label_percentages[label_columns]
    .sum(axis=1)
)

top_five_label_percentages[label_columns] = (
    top_five_label_percentages[label_columns]
    .div(label_totals, axis=0)
    .mul(100)
    .round(2)
)

top_five_label_percentages

label,state_clean,city_clean,drink,food,inside,menu,outside
0,FL,Tampa,8.68,51.78,28.69,0.85,10.01
1,IN,Indianapolis,8.35,52.72,30.04,1.04,7.85
2,LA,New Orleans,7.31,47.91,34.29,0.65,9.84
3,PA,Philadelphia,6.47,55.47,29.03,0.98,8.04
4,TN,Nashville,8.70,44.65,34.78,0.92,10.95


In [58]:
decision_columns = [
    "state_clean",
    "city_clean",
    "business_count",
    "total_reviews",
    "median_reviews",
    "businesses_with_photos",
    "photo_coverage_percentage",
    "total_photos",
    "median_photos_per_business",
    "p95_photos_per_business",
    "businesses_with_captioned_photos",
    "captioned_business_coverage_percentage",
    "captioned_photo_percentage",
    "unique_photo_labels"
]

location_decision_table = (
    top_five_location_images[
        decision_columns
    ]
    .copy()
)

location_decision_table

,state_clean,city_clean,business_count,total_reviews,median_reviews,businesses_with_photos,photo_coverage_percentage,total_photos,median_photos_per_business,p95_photos_per_business,businesses_with_captioned_photos,captioned_business_coverage_percentage,captioned_photo_percentage,unique_photo_labels
0,PA,Philadelphia,7692,751438,31.0,4038,52.50,25299,1.0,14.0,2904,37.75,50.32,5
1,FL,Tampa,4073,334036,30.0,2344,57.55,15642,1.0,17.0,1726,42.38,52.54,5
2,IN,Indianapolis,3905,281357,28.0,2090,53.52,11891,1.0,14.0,1425,36.49,48.32,5
3,TN,Nashville,3526,367155,34.0,1933,54.82,14093,1.0,19.0,1421,40.30,50.29,5
4,LA,New Orleans,3505,547106,41.0,1977,56.41,19456,1.0,25.0,1562,44.56,52.89,5


# Review Exploration

In [71]:
from pathlib import Path
import pandas as pd

data_dir = Path("YelpDataset")

review_path = (
    data_dir / "yelp_academic_dataset_review.json"
)

output_dir = Path(
    "processed_data/new_orleans_subset"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

selected_business_ids = set(
    selected_businesses["business_id"]
)

review_columns = [
    "review_id",
    "user_id",
    "business_id",
    "stars",
    "useful",
    "funny",
    "cool",
    "text",
    "date"
]

review_parts = []
retained_reviews = 0

for chunk_number, chunk in enumerate(
    pd.read_json(
        review_path,
        lines=True,
        chunksize=100_000
    )
):
    filtered_chunk = chunk.loc[
        chunk["business_id"].isin(
            selected_business_ids
        ),
        review_columns
    ].copy()

    if not filtered_chunk.empty:
        review_parts.append(filtered_chunk)
        retained_reviews += len(filtered_chunk)

    if chunk_number % 10 == 0:
        print(
            f"Processed chunk {chunk_number}; "
            f"retained {retained_reviews:,} reviews"
        )

selected_reviews = pd.concat(
    review_parts,
    ignore_index=True
)

selected_reviews["date"] = pd.to_datetime(
    selected_reviews["date"],
    errors="coerce"
)

selected_reviews = (
    selected_reviews
    .drop_duplicates(
        subset="review_id",
        keep="first"
    )
    .reset_index(drop=True)
)

selected_reviews["interaction_sentiment"] = np.select(
    [
        selected_reviews["stars"] >= 4,
        selected_reviews["stars"] <= 2
    ],
    [
        "positive",
        "negative"
    ],
    default="neutral"
)

# Optional Boolean columns for filtering later
selected_reviews["is_positive"] = (
    selected_reviews["interaction_sentiment"] == "positive"
)

selected_reviews["is_negative"] = (
    selected_reviews["interaction_sentiment"] == "negative"
)

selected_reviews["is_neutral"] = (
    selected_reviews["interaction_sentiment"] == "neutral"
)

print("Selected reviews:", len(selected_reviews))
print(
    "Unique users:",
    selected_reviews["user_id"].nunique()
)
print(
    "Reviewed businesses:",
    selected_reviews["business_id"].nunique()
)
print(
    "Positive reviews:",
    selected_reviews["is_positive"].sum()
)
print(
    "Negative reviews:",
    selected_reviews["is_negative"].sum()
)
print(
    "Neutral reviews:",
    selected_reviews["is_neutral"].sum()
)

Processed chunk 0; retained 10,265 reviews
Processed chunk 10; retained 94,389 reviews
Processed chunk 20; retained 165,173 reviews
Processed chunk 30; retained 250,081 reviews
Processed chunk 40; retained 328,264 reviews
Processed chunk 50; retained 416,402 reviews
Processed chunk 60; retained 496,215 reviews
Selected reviews: 559117
Unique users: 222493
Reviewed businesses: 3505
Positive reviews: 403225
Negative reviews: 92817
Neutral reviews: 63075


In [72]:
print("\nRating distribution:")
print(
    selected_reviews["stars"]
    .value_counts()
    .sort_index()
)

print("\nSentiment distribution:")
print(
    selected_reviews["interaction_sentiment"]
    .value_counts()
)


Rating distribution:
stars
1     52061
2     40756
3     63075
4    139229
5    263996
Name: count, dtype: int64

Sentiment distribution:
interaction_sentiment
positive    403225
negative     92817
neutral      63075
Name: count, dtype: int64


In [73]:
reviews_raw_path = (
    output_dir
    / "new_orleans_food_hospitality_reviews_raw.parquet"
)

selected_reviews.to_parquet(
    reviews_raw_path,
    index=False,
    engine="pyarrow"
)

print(
    f"Saved {len(selected_reviews):,} reviews to:\n"
    f"{reviews_raw_path.resolve()}"
)

Saved 559,117 reviews to:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/new_orleans_food_hospitality_reviews_raw.parquet


In [74]:
user_review_counts = (
    selected_reviews
    .groupby("user_id")
    .size()
)

business_review_counts = (
    selected_reviews
    .groupby("business_id")
    .size()
)

positive_reviews = selected_reviews.loc[
    selected_reviews["is_positive"]
].copy()

positive_user_counts = (
    positive_reviews
    .groupby("user_id")
    .size()
)

positive_business_counts = (
    positive_reviews
    .groupby("business_id")
    .size()
)

In [75]:
interaction_density_audit = pd.DataFrame({
    "metric": [
        "Total reviews",
        "Unique users",
        "Unique businesses",
        "Average reviews per user",
        "Median reviews per user",
        "Users with at least 2 reviews",
        "Users with at least 3 reviews",
        "Users with at least 5 reviews",
        "Users with at least 10 reviews",
        "Median reviews per business",
        "Businesses with at least 5 reviews",
        "Positive reviews",
        "Users with at least 3 positive reviews",
        "Users with at least 5 positive reviews",
        "Businesses with at least 5 positive reviews"
    ],
    "value": [
        len(selected_reviews),
        selected_reviews["user_id"].nunique(),
        selected_reviews["business_id"].nunique(),
        round(user_review_counts.mean(), 2),
        user_review_counts.median(),
        (user_review_counts >= 2).sum(),
        (user_review_counts >= 3).sum(),
        (user_review_counts >= 5).sum(),
        (user_review_counts >= 10).sum(),
        business_review_counts.median(),
        (business_review_counts >= 5).sum(),
        len(positive_reviews),
        (positive_user_counts >= 3).sum(),
        (positive_user_counts >= 5).sum(),
        (positive_business_counts >= 5).sum()
    ]
})

interaction_density_audit

,metric,value
0,Total reviews,559117.00
1,Unique users,222493.00
2,Unique businesses,3505.00
3,Average reviews per user,2.51
4,Median reviews per user,1.00
5,Users with at least 2 reviews,87840.00
6,Users with at least 3 reviews,50927.00
7,Users with at least 5 reviews,23476.00
8,Users with at least 10 reviews,6894.00
9,Median reviews per business,42.00


In [76]:
print("All reviews per user:")
print(user_review_counts.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))

print("\nPositive reviews per user:")
print(positive_user_counts.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))

print("\nReviews per business:")
print(business_review_counts.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))

All reviews per user:
count    222493.000000
mean          2.512964
std           7.037402
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
90%           5.000000
95%           7.000000
99%          18.000000
max         952.000000
dtype: float64

Positive reviews per user:
count    175667.000000
mean          2.295394
std           5.494515
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
90%           4.000000
95%           6.000000
99%          16.000000
max         731.000000
dtype: float64

Reviews per business:
count    3505.000000
mean      159.519829
std       413.862559
min         5.000000
25%        14.000000
50%        42.000000
75%       143.000000
90%       364.000000
95%       624.400000
99%      1914.160000
max      7673.000000
dtype: float64


In [77]:
positive_interactions = selected_reviews.loc[
    selected_reviews["is_positive"],
    [
        "review_id",
        "user_id",
        "business_id",
        "stars",
        "date"
    ]
].copy()

negative_interactions = selected_reviews.loc[
    selected_reviews["is_negative"],
    [
        "review_id",
        "user_id",
        "business_id",
        "stars",
        "date"
    ]
].copy()

neutral_interactions = selected_reviews.loc[
    selected_reviews["is_neutral"],
    [
        "review_id",
        "user_id",
        "business_id",
        "stars",
        "date"
    ]
].copy()

In [78]:
def iterative_k_core(
    interactions,
    min_user_interactions=5,
    min_business_interactions=5
):
    filtered = interactions.copy()
    iteration = 0

    while True:
        previous_rows = len(filtered)

        valid_users = (
            filtered["user_id"]
            .value_counts()
            .loc[
                lambda counts:
                counts >= min_user_interactions
            ]
            .index
        )

        filtered = filtered.loc[
            filtered["user_id"].isin(valid_users)
        ]

        valid_businesses = (
            filtered["business_id"]
            .value_counts()
            .loc[
                lambda counts:
                counts >= min_business_interactions
            ]
            .index
        )

        filtered = filtered.loc[
            filtered["business_id"].isin(
                valid_businesses
            )
        ]

        iteration += 1

        print(
            f"Iteration {iteration}: "
            f"{len(filtered):,} reviews, "
            f"{filtered['user_id'].nunique():,} users, "
            f"{filtered['business_id'].nunique():,} businesses"
        )

        if len(filtered) == previous_rows:
            break

    return (
        filtered
        .sort_values(
            ["user_id", "date", "review_id"]
        )
        .reset_index(drop=True)
    )

In [79]:
core_reviews = iterative_k_core(
    selected_reviews,
    min_user_interactions=5,
    min_business_interactions=5
)

Iteration 1: 257,069 reviews, 23,476 users, 3,027 businesses
Iteration 2: 256,520 reviews, 23,340 users, 3,025 businesses
Iteration 3: 256,512 reviews, 23,338 users, 3,025 businesses
Iteration 4: 256,512 reviews, 23,338 users, 3,025 businesses


In [80]:
core_reviews.to_parquet(
    output_dir
    / "new_orleans_reviews_5core.parquet",
    index=False
)

In [81]:
core_reviews = (
    core_reviews
    .sort_values(
        ["user_id", "date", "review_id"]
    )
    .reset_index(drop=True)
)

core_reviews["position_from_end"] = (
    core_reviews
    .groupby("user_id")
    .cumcount(ascending=False)
)

test_reviews = (
    core_reviews.loc[
        core_reviews["position_from_end"] == 0
    ]
    .drop(columns="position_from_end")
    .reset_index(drop=True)
)

validation_reviews = (
    core_reviews.loc[
        core_reviews["position_from_end"] == 1
    ]
    .drop(columns="position_from_end")
    .reset_index(drop=True)
)

train_reviews = (
    core_reviews.loc[
        core_reviews["position_from_end"] >= 2
    ]
    .drop(columns="position_from_end")
    .reset_index(drop=True)
)

print("Training reviews:", len(train_reviews))
print("Validation reviews:", len(validation_reviews))
print("Test reviews:", len(test_reviews))

Training reviews: 209836
Validation reviews: 23338
Test reviews: 23338


In [97]:
# train_reviews.to_parquet(
#     output_dir / "reviews_train.parquet",
#     index=False
# )

# validation_reviews.to_parquet(
#     output_dir / "reviews_validation.parquet",
#     index=False
# )

# test_reviews.to_parquet(
#     output_dir / "reviews_test.parquet",
#     index=False
# )

# Users Exploration

In [83]:
core_user_ids = set(
    core_reviews["user_id"]
)

user_path = (
    data_dir / "yelp_academic_dataset_user.json"
)

user_columns = [
    "user_id",
    "review_count",
    "yelping_since",
    "useful",
    "funny",
    "cool",
    "elite",
    "fans",
    "average_stars"
]

user_parts = []
retained_users = 0

for chunk_number, chunk in enumerate(
    pd.read_json(
        user_path,
        lines=True,
        chunksize=100_000
    )
):
    available_columns = [
        column
        for column in user_columns
        if column in chunk.columns
    ]

    filtered_chunk = chunk.loc[
        chunk["user_id"].isin(core_user_ids),
        available_columns
    ].copy()

    if not filtered_chunk.empty:
        user_parts.append(filtered_chunk)
        retained_users += len(filtered_chunk)

    if chunk_number % 5 == 0:
        print(
            f"Processed user chunk {chunk_number}; "
            f"retained {retained_users:,} users"
        )

selected_users = pd.concat(
    user_parts,
    ignore_index=True
)

selected_users = (
    selected_users
    .drop_duplicates(
        subset="user_id",
        keep="first"
    )
    .reset_index(drop=True)
)

selected_users["yelping_since"] = pd.to_datetime(
    selected_users["yelping_since"],
    errors="coerce"
)

print("Selected users:", len(selected_users))
print(
    "Expected core users:",
    len(core_user_ids)
)

Processed user chunk 0; retained 7,881 users
Processed user chunk 5; retained 19,762 users
Processed user chunk 10; retained 22,935 users
Processed user chunk 15; retained 23,331 users
Selected users: 23338
Expected core users: 23338


In [84]:
selected_users.to_parquet(
    output_dir
    / "new_orleans_users_5core.parquet",
    index=False
)

selected_users.to_csv(
    output_dir
    / "new_orleans_users_5core.csv",
    index=False,
    encoding="utf-8"
)

In [85]:
core_business_ids = set(
    core_reviews["business_id"]
)

core_businesses = (
    selected_businesses.loc[
        selected_businesses["business_id"].isin(
            core_business_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

core_photos = (
    selected_photos.loc[
        selected_photos["business_id"].isin(
            core_business_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print("Core businesses:", len(core_businesses))
print("Core photos:", len(core_photos))
print(
    "Core businesses with photos:",
    core_photos["business_id"].nunique()
)

Core businesses: 3025
Core photos: 19234
Core businesses with photos: 1889


In [86]:
core_businesses.to_parquet(
    output_dir
    / "new_orleans_businesses_5core.parquet",
    index=False
)

core_photos.to_parquet(
    output_dir
    / "new_orleans_photos_5core.parquet",
    index=False
)

In [87]:
split_sentiment_check = pd.DataFrame({
    "train": train_reviews["interaction_sentiment"].value_counts(),
    "validation": validation_reviews["interaction_sentiment"].value_counts(),
    "test": test_reviews["interaction_sentiment"].value_counts()
}).fillna(0).astype(int)

split_sentiment_check

,train,validation,test
interaction_sentiment,,,
negative,25300,2769,3024
neutral,31933,3185,2979
positive,152603,17384,17335


In [88]:
print("Validation rating distribution:")
print(validation_reviews["stars"].value_counts().sort_index())

print("\nTest rating distribution:")
print(test_reviews["stars"].value_counts().sort_index())

Validation rating distribution:
stars
1     1221
2     1548
3     3185
4     7021
5    10363
Name: count, dtype: int64

Test rating distribution:
stars
1     1454
2     1570
3     2979
4     6666
5    10669
Name: count, dtype: int64


In [89]:
all_reviews_5core = core_reviews.copy()

print("All-review 5-core:", all_reviews_5core.shape)

All-review 5-core: (256512, 14)


# Create the final location subset

In [59]:
SELECTED_STATE = "LA"
SELECTED_CITY = "New Orleans"

selected_businesses = (
    target_businesses.loc[
        target_businesses["state_clean"].eq(SELECTED_STATE)
        & target_businesses["city_clean"].eq(SELECTED_CITY)
    ]
    .copy()
    .reset_index(drop=True)
)

selected_business_ids = set(
    selected_businesses["business_id"]
)

selected_photos = (
    photos_df_clean.loc[
        photos_df_clean["business_id"].isin(
            selected_business_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print("Selected businesses:", len(selected_businesses))
print("Selected photos:", len(selected_photos))
print(
    "Businesses with photos:",
    selected_photos["business_id"].nunique()
)

Selected businesses: 3505
Selected photos: 19456
Businesses with photos: 1977


In [60]:
# Preserve open status without filtering yet

selected_businesses["is_operational_at_snapshot"] = (
    selected_businesses["is_open"].eq(1)
)

selection_summary = pd.Series({
    "Total businesses": len(selected_businesses),
    "Open at snapshot": (
        selected_businesses["is_operational_at_snapshot"].sum()
    ),
    "Closed at snapshot": (
        ~selected_businesses["is_operational_at_snapshot"]
    ).sum(),
    "Businesses with photos": (
        selected_photos["business_id"].nunique()
    ),
    "Total photos": len(selected_photos),
    "Total recorded reviews": (
        selected_businesses["review_count"].sum()
    )
})

selection_summary

Total businesses            3505
Open at snapshot            2374
Closed at snapshot          1131
Businesses with photos      1977
Total photos               19456
Total recorded reviews    547106
dtype: int64

## Save Subset in CSV

In [ ]:
# from pathlib import Path
# import json
# import pandas as pd

# output_dir = Path("processed_data/new_orleans_subset")
# output_dir.mkdir(parents=True, exist_ok=True)


# def serialise_nested_value(value):
#     """Convert lists and dictionaries into valid JSON text for CSV storage."""
#     if isinstance(value, (list, dict)):
#         return json.dumps(value, ensure_ascii=False)

#     return value

In [ ]:
# selected_businesses_csv = selected_businesses.copy()

# nested_business_columns = [
#     "attributes",
#     "hours",
#     "category_list",
#     "matched_target_categories"
# ]

# for column in nested_business_columns:
#     if column in selected_businesses_csv.columns:
#         selected_businesses_csv[column] = (
#             selected_businesses_csv[column]
#             .apply(serialise_nested_value)
#         )

# business_csv_path = (
#     output_dir
#     / "new_orleans_food_hospitality_businesses.csv"
# )

# selected_businesses_csv.to_csv(
#     business_csv_path,
#     index=False,
#     encoding="utf-8"
# )

# print(
#     f"Saved {len(selected_businesses_csv):,} businesses to:"
#     f"\n{business_csv_path.resolve()}"
# )

Saved 3,505 businesses to:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/new_orleans_food_hospitality_businesses.csv


In [ ]:
# photo_csv_path = (
#     output_dir
#     / "new_orleans_food_hospitality_photos.csv"
# )

# selected_photos.to_csv(
#     photo_csv_path,
#     index=False,
#     encoding="utf-8"
# )

# print(
#     f"Saved {len(selected_photos):,} photo records to:"
#     f"\n{photo_csv_path.resolve()}"
# )

Saved 19,456 photo records to:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/new_orleans_food_hospitality_photos.csv


In [ ]:
# comparison_csv_path = (
#     output_dir
#     / "top_five_location_image_comparison.csv"
# )

# location_decision_table.to_csv(
#     comparison_csv_path,
#     index=False,
#     encoding="utf-8"
# )

# print(
#     f"Saved location comparison to:"
#     f"\n{comparison_csv_path.resolve()}"
# )

Saved location comparison to:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/top_five_location_image_comparison.csv


In [ ]:
# for path in [
#     business_csv_path,
#     photo_csv_path,
#     comparison_csv_path
# ]:
#     print(
#         path.name,
#         "| exists:", path.exists(),
#         "| size:", round(path.stat().st_size / 1_000_000, 2), "MB"
#     )

new_orleans_food_hospitality_businesses.csv | exists: True | size: 3.99 MB
new_orleans_food_hospitality_photos.csv | exists: True | size: 1.79 MB
top_five_location_image_comparison.csv | exists: True | size: 0.0 MB


## Save in Parquet

In [66]:
output_dir = Path("processed_data/new_orleans_subset")
output_dir.mkdir(parents=True, exist_ok=True)


def serialise_nested_value(value):
    """Convert nested Python objects into JSON strings."""
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(
            list(value) if isinstance(value, set) else value,
            ensure_ascii=False,
            default=str
        )

    return value

In [67]:
selected_businesses_parquet = selected_businesses.copy()

nested_business_columns = [
    "attributes",
    "hours",
    "category_list",
    "matched_target_categories"
]

for column in nested_business_columns:
    if column in selected_businesses_parquet.columns:
        selected_businesses_parquet[column] = (
            selected_businesses_parquet[column]
            .apply(serialise_nested_value)
        )

business_parquet_path = (
    output_dir
    / "new_orleans_food_hospitality_businesses.parquet"
)

selected_businesses_parquet.to_parquet(
    business_parquet_path,
    index=False,
    engine="pyarrow"
)

print(
    f"Saved {len(selected_businesses_parquet):,} businesses to:\n"
    f"{business_parquet_path.resolve()}"
)

Saved 3,505 businesses to:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/new_orleans_food_hospitality_businesses.parquet


In [68]:
photo_parquet_path = (
    output_dir
    / "new_orleans_food_hospitality_photos.parquet"
)

selected_photos.to_parquet(
    photo_parquet_path,
    index=False,
    engine="pyarrow"
)

print(
    f"Saved {len(selected_photos):,} photo records to:\n"
    f"{photo_parquet_path.resolve()}"
)

Saved 19,456 photo records to:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/new_orleans_food_hospitality_photos.parquet


In [69]:
business_check = pd.read_parquet(business_parquet_path)
photo_check = pd.read_parquet(photo_parquet_path)

print("Businesses:", business_check.shape)
print("Photos:", photo_check.shape)

Businesses: (3505, 20)
Photos: (19456, 6)


In [70]:
business_check.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours,city_clean,state_clean,category_list,matched_target_categories,is_target_business,is_operational_at_snapshot
0,w_AMNoI1iG9eay7ncmc67w,River 127,100 Iberville St,New Orleans,LA,70130,29.951359,-90.064672,3.0,12,1,"{""BusinessAcceptsCreditCards"": ""True"", ""WiFi"":...","Event Planning & Services, Hotels, Hotels & Tr...",None,New Orleans,LA,"[""Event Planning & Services"", ""Hotels"", ""Hotel...","[""Hotels""]",True,True
1,YNjyv0gfOr2g8lbmUpTnKg,Copper Vine,1001 Poydras St,New Orleans,LA,70112,29.950647,-90.074427,4.5,350,1,"{""NoiseLevel"": ""u'average'"", ""RestaurantsGoodF...","Nightlife, Pubs, Event Planning & Services, Wi...","{""Monday"": ""0:0-0:0"", ""Tuesday"": ""11:0-21:0"", ...",New Orleans,LA,"[""Nightlife"", ""Pubs"", ""Event Planning & Servic...","[""Bars"", ""Nightlife"", ""Pubs"", ""Restaurants"", ""...",True,True
2,J_ksUDPpzPwfTGtI4zTRnQ,Riverview Room,600 Decatur St,New Orleans,LA,70130,29.955925,-90.062962,4.5,7,1,"{""BusinessAcceptsCreditCards"": ""True"", ""WiFi"":...","Event Planning & Services, Caterers, Party & E...","{""Monday"": ""9:0-16:0"", ""Tuesday"": ""9:0-16:0"", ...",New Orleans,LA,"[""Event Planning & Services"", ""Caterers"", ""Par...","[""Caterers""]",True,True
3,TLZ3-eDPLhUzfsWO4ad6Ug,Mahony's Po-Boys & Seafood,901 Iberville St,New Orleans,LA,70112,29.955415,-90.070062,4.0,382,1,"{""RestaurantsGoodForGroups"": ""True"", ""DogsAllo...","Restaurants, Seafood, Cajun/Creole","{""Monday"": ""0:0-0:0"", ""Thursday"": ""15:0-20:0"",...",New Orleans,LA,"[""Restaurants"", ""Seafood"", ""Cajun/Creole""]","[""Restaurants""]",True,True
4,FRYkg_JvsWU9xIXZsEZcVA,Altamura,2127 Prytania St,New Orleans,LA,70115,29.933388,-90.079498,3.5,27,0,"{""Alcohol"": ""'full_bar'"", ""OutdoorSeating"": ""T...","Cocktail Bars, Italian, Nightlife, Seafood, Ba...","{""Monday"": ""17:0-22:0"", ""Wednesday"": ""17:0-22:...",New Orleans,LA,"[""Cocktail Bars"", ""Italian"", ""Nightlife"", ""Sea...","[""Bars"", ""Cocktail Bars"", ""Nightlife"", ""Restau...",True,False


# Positive 5-core dataset

In [90]:
positive_reviews = (
    selected_reviews.loc[
        selected_reviews["interaction_sentiment"] == "positive"
    ]
    .copy()
    .reset_index(drop=True)
)

print("Positive reviews before 5-core:", len(positive_reviews))
print(
    "Users before 5-core:",
    positive_reviews["user_id"].nunique()
)
print(
    "Businesses before 5-core:",
    positive_reviews["business_id"].nunique()
)

Positive reviews before 5-core: 403225
Users before 5-core: 175667
Businesses before 5-core: 3472


In [91]:
positive_reviews_5core = iterative_k_core(
    positive_reviews,
    min_user_interactions=5,
    min_business_interactions=5
)

Iteration 1: 161,283 reviews, 15,621 users, 2,550 businesses
Iteration 2: 160,524 reviews, 15,428 users, 2,549 businesses
Iteration 3: 160,524 reviews, 15,428 users, 2,549 businesses


In [92]:
positive_core_summary = pd.Series({
    "Positive interactions": len(positive_reviews_5core),
    "Users": positive_reviews_5core["user_id"].nunique(),
    "Businesses": positive_reviews_5core["business_id"].nunique(),
    "Minimum interactions per user": (
        positive_reviews_5core
        .groupby("user_id")
        .size()
        .min()
    ),
    "Minimum interactions per business": (
        positive_reviews_5core
        .groupby("business_id")
        .size()
        .min()
    )
})

positive_core_summary

Positive interactions                160524
Users                                 15428
Businesses                             2549
Minimum interactions per user             5
Minimum interactions per business         5
dtype: int64

In [93]:
positive_core_path = (
    output_dir
    / "new_orleans_positive_reviews_5core.parquet"
)

positive_reviews_5core.to_parquet(
    positive_core_path,
    index=False,
    engine="pyarrow"
)

print(
    f"Saved {len(positive_reviews_5core):,} positive interactions to:\n"
    f"{positive_core_path.resolve()}"
)

Saved 160,524 positive interactions to:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/new_orleans_positive_reviews_5core.parquet


In [94]:
positive_core_check = pd.read_parquet(
    positive_core_path
)

print("Shape:", positive_core_check.shape)
print(
    "Users:",
    positive_core_check["user_id"].nunique()
)
print(
    "Businesses:",
    positive_core_check["business_id"].nunique()
)

Shape: (160524, 13)
Users: 15428
Businesses: 2549


# Dataset Registry

In [110]:
dataset_registry = pd.DataFrame([
    {
        "dataset": "new_orleans_food_hospitality_businesses",
        "records": 3505,
        "users": None,
        "businesses": 3505,
        "purpose": "Full domain and location-specific business subset",
        "project_use": "Metadata, auditing, cold-start analysis and subset source",
        "status": "Final selected raw subset"
    },
    {
        "dataset": "new_orleans_food_hospitality_photos",
        "records": 19456,
        "users": None,
        "businesses": 1977,
        "purpose": "All photographs linked to selected businesses",
        "project_use": "Image and caption embeddings, visual explanations",
        "status": "Final selected raw subset"
    },
    {
        "dataset": "new_orleans_food_hospitality_reviews_raw",
        "records": 559117,
        "users": 222493,
        "businesses": 3505,
        "purpose": "Complete review and interaction evidence",
        "project_use": "Text modelling, sentiment, filtering and explanations",
        "status": "Final selected raw subset"
    },
    {
        "dataset": "new_orleans_all_reviews_5core",
        "records": 256512,
        "users": 23338,
        "businesses": 3025,
        "purpose": "Connected review dataset containing all ratings",
        "project_use": "Review embeddings, business representation and evidence",
        "status": "Modelling dataset"
    },
    {
        "dataset": "all_review_train",
        "records": 209836,
        "users": 23338,
        "businesses": None,
        "purpose": "Temporal split containing all rating types",
        "project_use": "Diagnostic and text experiments only",
        "status": "Not final ranking split"
    },
    {
        "dataset": "all_review_validation",
        "records": 23338,
        "users": 23338,
        "businesses": None,
        "purpose": "Held-out interactions containing mixed sentiment",
        "project_use": "Diagnostic only",
        "status": "Not final ranking split"
    },
    {
        "dataset": "all_review_test",
        "records": 23338,
        "users": 23338,
        "businesses": None,
        "purpose": "Held-out interactions containing mixed sentiment",
        "project_use": "Diagnostic only",
        "status": "Not final ranking split"
    },
    {
        "dataset": "new_orleans_users_5core",
        "records": 23338,
        "users": 23338,
        "businesses": None,
        "purpose": "User metadata for all-review-core users",
        "project_use": "User audit and supplementary features",
        "status": "Supporting dataset"
    },
    {
        "dataset": "new_orleans_businesses_5core",
        "records": 3025,
        "users": None,
        "businesses": 3025,
        "purpose": "Business metadata aligned with all-review core",
        "project_use": "Business nodes and text representation",
        "status": "Supporting dataset"
    },
    {
        "dataset": "new_orleans_photos_5core",
        "records": 19234,
        "users": None,
        "businesses": 1889,
        "purpose": "Photos aligned with all-review core",
        "project_use": "Multimodal experiments and image selection",
        "status": "Supporting dataset"
    },
    {
        "dataset": "new_orleans_positive_reviews_5core",
        "records": 160524,
        "users": 15428,
        "businesses": 2549,
        "purpose": "Connected positive-preference interactions",
        "project_use": "Personalisation, KGRec and ranking evaluation",
        "status": "Primary recommendation dataset"
    }
])

registry_path = output_dir / "dataset_registry.csv"

dataset_registry.to_csv(
    registry_path,
    index=False,
    encoding="utf-8"
)

dataset_registry

,dataset,records,users,businesses,purpose,project_use,status
0,new_orleans_food_hospitality_businesses,3505,NaN,3505.0,Full domain and location-specific business subset,"Metadata, auditing, cold-start analysis and su...",Final selected raw subset
1,new_orleans_food_hospitality_photos,19456,NaN,1977.0,All photographs linked to selected businesses,"Image and caption embeddings, visual explanations",Final selected raw subset
2,new_orleans_food_hospitality_reviews_raw,559117,222493.0,3505.0,Complete review and interaction evidence,"Text modelling, sentiment, filtering and expla...",Final selected raw subset
3,new_orleans_all_reviews_5core,256512,23338.0,3025.0,Connected review dataset containing all ratings,"Review embeddings, business representation and...",Modelling dataset
4,all_review_train,209836,23338.0,NaN,Temporal split containing all rating types,Diagnostic and text experiments only,Not final ranking split
5,all_review_validation,23338,23338.0,NaN,Held-out interactions containing mixed sentiment,Diagnostic only,Not final ranking split
6,all_review_test,23338,23338.0,NaN,Held-out interactions containing mixed sentiment,Diagnostic only,Not final ranking split
7,new_orleans_users_5core,23338,23338.0,NaN,User metadata for all-review-core users,User audit and supplementary features,Supporting dataset
8,new_orleans_businesses_5core,3025,NaN,3025.0,Business metadata aligned with all-review core,Business nodes and text representation,Supporting dataset
9,new_orleans_photos_5core,19234,NaN,1889.0,Photos aligned with all-review core,Multimodal experiments and image selection,Supporting dataset


In [114]:
from pathlib import Path
import pandas as pd

output_dir = Path("processed_data/new_orleans_subset")

# Reload saved datasets so this stage is reproducible after a restart
positive_reviews_5core = pd.read_parquet(
    output_dir / "new_orleans_positive_reviews_5core.parquet"
)

selected_users = pd.read_parquet(
    output_dir / "new_orleans_users_5core.parquet"
)

selected_businesses = pd.read_parquet(
    output_dir / "new_orleans_food_hospitality_businesses.parquet"
)

selected_photos = pd.read_parquet(
    output_dir / "new_orleans_food_hospitality_photos.parquet"
)

positive_user_ids = set(
    positive_reviews_5core["user_id"]
)

positive_business_ids = set(
    positive_reviews_5core["business_id"]
)

integrity_check = pd.Series({
    "Positive interactions":
        len(positive_reviews_5core),

    "Unique users":
        positive_reviews_5core["user_id"].nunique(),

    "Unique businesses":
        positive_reviews_5core["business_id"].nunique(),

    "Duplicate review IDs":
        positive_reviews_5core["review_id"].duplicated().sum(),

    "Duplicate user-business pairs":
        positive_reviews_5core.duplicated(
            subset=["user_id", "business_id"]
        ).sum(),

    "Missing dates":
        positive_reviews_5core["date"].isna().sum(),

    "Reviews below 4 stars":
        (positive_reviews_5core["stars"] < 4).sum(),

    "Minimum interactions per user":
        positive_reviews_5core
        .groupby("user_id")
        .size()
        .min(),

    "Minimum interactions per business":
        positive_reviews_5core
        .groupby("business_id")
        .size()
        .min(),

    "Positive users missing from user data":
        len(
            positive_user_ids
            - set(selected_users["user_id"])
        ),

    "Positive businesses missing from business data":
        len(
            positive_business_ids
            - set(selected_businesses["business_id"])
        ),

    "Photos linked to positive-core businesses":
        selected_photos["business_id"]
        .isin(positive_business_ids)
        .sum(),

    "Positive-core businesses with photos":
        selected_photos.loc[
            selected_photos["business_id"].isin(
                positive_business_ids
            ),
            "business_id"
        ].nunique()
})

integrity_check

Positive interactions                             160524
Unique users                                       15428
Unique businesses                                   2549
Duplicate review IDs                                   0
Duplicate user-business pairs                       5498
Missing dates                                          0
Reviews below 4 stars                                  0
Minimum interactions per user                          5
Minimum interactions per business                      5
Positive users missing from user data                  0
Positive businesses missing from business data         0
Photos linked to positive-core businesses          18850
Positive-core businesses with photos                1738
dtype: int64

In [115]:
from pathlib import Path

obsolete_files = [
    output_dir / "reviews_train.parquet",
    output_dir / "reviews_validation.parquet",
    output_dir / "reviews_test.parquet"
]

for file_path in obsolete_files:
    if file_path.exists():
        file_path.unlink()
        print("Deleted:", file_path.name)
    else:
        print("Not found:", file_path.name)

Deleted: reviews_train.parquet
Deleted: reviews_validation.parquet
Deleted: reviews_test.parquet


In [96]:
from pathlib import Path
from datetime import datetime
import shutil

output_dir = Path(
    "processed_data/new_orleans_subset"
)

archive_dir = (
    output_dir / "_archive_before_final_preparation"
)

archive_dir.mkdir(
    parents=True,
    exist_ok=True
)

files_to_archive = [
    # Duplicate CSV copies
    "new_orleans_food_hospitality_businesses.csv",
    "new_orleans_food_hospitality_photos.csv",
    "new_orleans_users_5core.csv",

    # Intermediate all-review 5-core datasets
    "new_orleans_businesses_5core.parquet",
    "new_orleans_photos_5core.parquet",
    "new_orleans_reviews_5core.parquet",

    # Uncorrected positive 5-core dataset
    "new_orleans_positive_reviews_5core.parquet",

    # Obsolete mixed-sentiment splits
    "reviews_train.parquet",
    "reviews_validation.parquet",
    "reviews_test.parquet",
]

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for filename in files_to_archive:
    source_path = output_dir / filename

    if not source_path.exists():
        print(f"Not found in active folder: {filename}")
        continue

    destination_path = archive_dir / filename

    # Preserve an existing archived copy
    if destination_path.exists():
        destination_path = (
            archive_dir
            / f"{source_path.stem}_from_active_{timestamp}{source_path.suffix}"
        )

        counter = 1

        while destination_path.exists():
            destination_path = (
                archive_dir
                / (
                    f"{source_path.stem}_from_active_"
                    f"{timestamp}_{counter}"
                    f"{source_path.suffix}"
                )
            )
            counter += 1

    shutil.move(
        str(source_path),
        str(destination_path)
    )

    print(
        f"Archived: {filename} "
        f"→ {destination_path.name}"
    )

print("\nActive files remaining:")

active_files = sorted(
    file_path.name
    for file_path in output_dir.iterdir()
    if file_path.is_file()
)

for filename in active_files:
    print("-", filename)

Archived: new_orleans_food_hospitality_businesses.csv → new_orleans_food_hospitality_businesses_from_active_20260803_133113.csv
Archived: new_orleans_food_hospitality_photos.csv → new_orleans_food_hospitality_photos_from_active_20260803_133113.csv
Archived: new_orleans_users_5core.csv → new_orleans_users_5core_from_active_20260803_133113.csv
Archived: new_orleans_businesses_5core.parquet → new_orleans_businesses_5core_from_active_20260803_133113.parquet
Archived: new_orleans_photos_5core.parquet → new_orleans_photos_5core_from_active_20260803_133113.parquet
Archived: new_orleans_reviews_5core.parquet → new_orleans_reviews_5core_from_active_20260803_133113.parquet
Archived: new_orleans_positive_reviews_5core.parquet → new_orleans_positive_reviews_5core_from_active_20260803_133113.parquet
Archived: reviews_train.parquet → reviews_train.parquet
Archived: reviews_validation.parquet → reviews_validation.parquet
Archived: reviews_test.parquet → reviews_test.parquet

Active files remaining:
-

In [98]:
from pathlib import Path
import pandas as pd

# Folder containing the cleaned active datasets
output_dir = Path("processed_data/new_orleans_subset")

required_files = {
    "businesses": (
        output_dir
        / "new_orleans_food_hospitality_businesses.parquet"
    ),
    "photos": (
        output_dir
        / "new_orleans_food_hospitality_photos.parquet"
    ),
    "reviews": (
        output_dir
        / "new_orleans_food_hospitality_reviews_raw.parquet"
    ),
    "users": (
        output_dir
        / "new_orleans_users_5core.parquet"
    ),
    "location_comparison": (
        output_dir
        / "top_five_location_image_comparison.csv"
    )
}

# Confirm that every required checkpoint exists
missing_files = [
    str(file_path)
    for file_path in required_files.values()
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following required files are missing:\n"
        + "\n".join(missing_files)
    )

print("All required checkpoint files were found.")

# Reload the saved datasets
selected_businesses = pd.read_parquet(
    required_files["businesses"]
)

selected_photos = pd.read_parquet(
    required_files["photos"]
)

selected_reviews = pd.read_parquet(
    required_files["reviews"]
)

selected_users = pd.read_parquet(
    required_files["users"]
)

location_comparison = pd.read_csv(
    required_files["location_comparison"]
)

# Ensure dates are available for temporal processing later
selected_reviews["date"] = pd.to_datetime(
    selected_reviews["date"],
    errors="coerce"
)

checkpoint_summary = pd.DataFrame({
    "dataset": [
        "Selected businesses",
        "Selected photos",
        "Raw selected reviews",
        "Available user metadata",
        "Location comparison"
    ],
    "rows": [
        len(selected_businesses),
        len(selected_photos),
        len(selected_reviews),
        len(selected_users),
        len(location_comparison)
    ],
    "unique_users": [
        None,
        None,
        selected_reviews["user_id"].nunique(),
        selected_users["user_id"].nunique(),
        None
    ],
    "unique_businesses": [
        selected_businesses["business_id"].nunique(),
        selected_photos["business_id"].nunique(),
        selected_reviews["business_id"].nunique(),
        None,
        None
    ]
})

checkpoint_summary

All required checkpoint files were found.


,dataset,rows,unique_users,unique_businesses
0,Selected businesses,3505,NaN,3505.0
1,Selected photos,19456,NaN,1977.0
2,Raw selected reviews,559117,222493.0,3505.0
3,Available user metadata,23338,23338.0,NaN
4,Location comparison,5,NaN,NaN


## Stage B1 — Create Unique Latest User–Business Interactions

A user may review the same business more than once. For recommendation
modelling, the latest review is retained so that each user–business pair
represents one current interaction.

In [99]:
# Confirm that temporal deduplication is safe
missing_dates = selected_reviews["date"].isna().sum()

if missing_dates > 0:
    raise ValueError(
        f"{missing_dates:,} reviews have missing dates. "
        "These must be handled before selecting the latest review."
    )

original_review_count = len(selected_reviews)

original_unique_pairs = (
    selected_reviews[
        ["user_id", "business_id"]
    ]
    .drop_duplicates()
    .shape[0]
)

# Sort reviews chronologically within each user-business pair
latest_interactions = (
    selected_reviews
    .sort_values(
        [
            "user_id",
            "business_id",
            "date",
            "review_id"
        ]
    )
    .drop_duplicates(
        subset=["user_id", "business_id"],
        keep="last"
    )
    .reset_index(drop=True)
)

remaining_duplicate_pairs = (
    latest_interactions
    .duplicated(
        subset=["user_id", "business_id"]
    )
    .sum()
)

latest_interaction_summary = pd.Series({
    "Original review rows":
        original_review_count,

    "Original unique user-business pairs":
        original_unique_pairs,

    "Latest interaction rows":
        len(latest_interactions),

    "Repeated review rows removed":
        original_review_count - len(latest_interactions),

    "Remaining duplicate user-business pairs":
        remaining_duplicate_pairs,

    "Unique users":
        latest_interactions["user_id"].nunique(),

    "Unique businesses":
        latest_interactions["business_id"].nunique(),

    "Earliest retained interaction":
        latest_interactions["date"].min(),

    "Latest retained interaction":
        latest_interactions["date"].max()
})

latest_interaction_summary

Original review rows                                    559117
Original unique user-business pairs                     547133
Latest interaction rows                                 547133
Repeated review rows removed                             11984
Remaining duplicate user-business pairs                      0
Unique users                                            222493
Unique businesses                                         3505
Earliest retained interaction              2005-03-14 18:07:51
Latest retained interaction                2022-01-19 19:47:59
dtype: object

In [100]:
latest_rating_distribution = (
    latest_interactions["stars"]
    .value_counts()
    .sort_index()
    .rename_axis("stars")
    .to_frame("interactions")
)

latest_rating_distribution["percentage"] = (
    latest_rating_distribution["interactions"]
    / len(latest_interactions)
    * 100
).round(2)

latest_rating_distribution

,interactions,percentage
stars,,
1,50872,9.30
2,39881,7.29
3,61727,11.28
4,136191,24.89
5,258462,47.24


## Stage B2 — Create and Inspect Positive-Preference Interactions

The latest review for each user–business pair is classified by rating.
Four- and five-star interactions are treated as positive preferences for
personalised recommendation. Neutral and negative interactions are retained
separately for textual evidence and analysis.

In [101]:
import numpy as np

latest_interactions = latest_interactions.copy()

latest_interactions["interaction_sentiment"] = np.select(
    [
        latest_interactions["stars"] >= 4,
        latest_interactions["stars"] <= 2
    ],
    [
        "positive",
        "negative"
    ],
    default="neutral"
)

positive_interactions = (
    latest_interactions.loc[
        latest_interactions["interaction_sentiment"] == "positive"
    ]
    .copy()
    .reset_index(drop=True)
)

neutral_interactions = (
    latest_interactions.loc[
        latest_interactions["interaction_sentiment"] == "neutral"
    ]
    .copy()
    .reset_index(drop=True)
)

negative_interactions = (
    latest_interactions.loc[
        latest_interactions["interaction_sentiment"] == "negative"
    ]
    .copy()
    .reset_index(drop=True)
)

sentiment_summary = pd.DataFrame({
    "interaction_type": [
        "Positive",
        "Neutral",
        "Negative"
    ],
    "rating_definition": [
        "4–5 stars",
        "3 stars",
        "1–2 stars"
    ],
    "interactions": [
        len(positive_interactions),
        len(neutral_interactions),
        len(negative_interactions)
    ],
    "unique_users": [
        positive_interactions["user_id"].nunique(),
        neutral_interactions["user_id"].nunique(),
        negative_interactions["user_id"].nunique()
    ],
    "unique_businesses": [
        positive_interactions["business_id"].nunique(),
        neutral_interactions["business_id"].nunique(),
        negative_interactions["business_id"].nunique()
    ]
})

sentiment_summary["percentage"] = (
    sentiment_summary["interactions"]
    / len(latest_interactions)
    * 100
).round(2)

sentiment_summary

,interaction_type,rating_definition,interactions,unique_users,unique_businesses,percentage
0,Positive,4–5 stars,394653,175588,3472,72.13
1,Neutral,3 stars,61727,37080,2922,11.28
2,Negative,1–2 stars,90753,68679,3127,16.59


In [102]:
positive_user_counts = (
    positive_interactions
    .groupby("user_id")
    .size()
)

positive_business_counts = (
    positive_interactions
    .groupby("business_id")
    .size()
)

positive_interaction_summary = pd.Series({
    "Positive interactions":
        len(positive_interactions),

    "Unique users":
        positive_interactions["user_id"].nunique(),

    "Unique businesses":
        positive_interactions["business_id"].nunique(),

    "Duplicate review IDs":
        positive_interactions["review_id"]
        .duplicated()
        .sum(),

    "Duplicate user-business pairs":
        positive_interactions
        .duplicated(
            subset=["user_id", "business_id"]
        )
        .sum(),

    "Missing dates":
        positive_interactions["date"]
        .isna()
        .sum(),

    "Interactions below 4 stars":
        (
            positive_interactions["stars"] < 4
        ).sum(),

    "Average positive interactions per user":
        round(positive_user_counts.mean(), 2),

    "Median positive interactions per user":
        positive_user_counts.median(),

    "Users with at least 3 positive interactions":
        (positive_user_counts >= 3).sum(),

    "Users with at least 5 positive interactions":
        (positive_user_counts >= 5).sum(),

    "Businesses with at least 5 positive interactions":
        (positive_business_counts >= 5).sum()
})

positive_interaction_summary

Positive interactions                               394653.00
Unique users                                        175588.00
Unique businesses                                     3472.00
Duplicate review IDs                                     0.00
Duplicate user-business pairs                            0.00
Missing dates                                            0.00
Interactions below 4 stars                               0.00
Average positive interactions per user                   2.25
Median positive interactions per user                    1.00
Users with at least 3 positive interactions          36180.00
Users with at least 5 positive interactions          15196.00
Businesses with at least 5 positive interactions      3068.00
dtype: float64

## Stage B3 — Apply the Iterative Positive 5-Core Filter

The positive-preference interactions are iteratively filtered until every
remaining user has interacted positively with at least five distinct
businesses and every remaining business has received positive interactions
from at least five distinct users.

In [103]:
def iterative_positive_k_core(
    interactions,
    min_user_interactions=5,
    min_business_interactions=5
):
    """
    Iteratively filter a user-business interaction table.

    The process stops when:
    - every user has at least min_user_interactions distinct businesses;
    - every business has at least min_business_interactions distinct users.
    """

    filtered = interactions.copy()
    iteration = 0

    while True:
        iteration += 1
        rows_before = len(filtered)

        # Retain users with enough distinct positive business interactions
        user_counts = (
            filtered
            .groupby("user_id")["business_id"]
            .nunique()
        )

        valid_users = user_counts.loc[
            user_counts >= min_user_interactions
        ].index

        filtered = filtered.loc[
            filtered["user_id"].isin(valid_users)
        ].copy()

        # Retain businesses with enough distinct positive users
        business_counts = (
            filtered
            .groupby("business_id")["user_id"]
            .nunique()
        )

        valid_businesses = business_counts.loc[
            business_counts >= min_business_interactions
        ].index

        filtered = filtered.loc[
            filtered["business_id"].isin(valid_businesses)
        ].copy()

        print(
            f"Iteration {iteration}: "
            f"{len(filtered):,} positive interactions, "
            f"{filtered['user_id'].nunique():,} users, "
            f"{filtered['business_id'].nunique():,} businesses"
        )

        # No rows were removed, so the dataset has stabilised
        if len(filtered) == rows_before:
            break

    return (
        filtered
        .sort_values(
            ["user_id", "date", "business_id", "review_id"]
        )
        .reset_index(drop=True)
    )

In [104]:
positive_reviews_5core = iterative_positive_k_core(
    positive_interactions,
    min_user_interactions=5,
    min_business_interactions=5
)

Iteration 1: 153,029 positive interactions, 15,196 users, 2,518 businesses
Iteration 2: 152,215 positive interactions, 14,991 users, 2,516 businesses
Iteration 3: 152,215 positive interactions, 14,991 users, 2,516 businesses


In [105]:
final_user_counts = (
    positive_reviews_5core
    .groupby("user_id")["business_id"]
    .nunique()
)

final_business_counts = (
    positive_reviews_5core
    .groupby("business_id")["user_id"]
    .nunique()
)

positive_5core_summary = pd.Series({
    "Positive interactions":
        len(positive_reviews_5core),

    "Unique users":
        positive_reviews_5core["user_id"].nunique(),

    "Unique businesses":
        positive_reviews_5core["business_id"].nunique(),

    "Duplicate review IDs":
        positive_reviews_5core["review_id"]
        .duplicated()
        .sum(),

    "Duplicate user-business pairs":
        positive_reviews_5core
        .duplicated(
            subset=["user_id", "business_id"]
        )
        .sum(),

    "Missing dates":
        positive_reviews_5core["date"]
        .isna()
        .sum(),

    "Interactions below 4 stars":
        (
            positive_reviews_5core["stars"] < 4
        ).sum(),

    "Minimum distinct businesses per user":
        final_user_counts.min(),

    "Minimum distinct users per business":
        final_business_counts.min(),

    "Average positive interactions per user":
        round(final_user_counts.mean(), 2),

    "Median positive interactions per user":
        final_user_counts.median(),

    "Users retained from positive dataset (%)":
        round(
            positive_reviews_5core["user_id"].nunique()
            / positive_interactions["user_id"].nunique()
            * 100,
            2
        ),

    "Businesses retained from positive dataset (%)":
        round(
            positive_reviews_5core["business_id"].nunique()
            / positive_interactions["business_id"].nunique()
            * 100,
            2
        )
})

positive_5core_summary

Positive interactions                            152215.00
Unique users                                      14991.00
Unique businesses                                  2516.00
Duplicate review IDs                                  0.00
Duplicate user-business pairs                         0.00
Missing dates                                         0.00
Interactions below 4 stars                            0.00
Minimum distinct businesses per user                  5.00
Minimum distinct users per business                   5.00
Average positive interactions per user               10.15
Median positive interactions per user                 7.00
Users retained from positive dataset (%)              8.54
Businesses retained from positive dataset (%)        72.47
dtype: float64

## Stage B4 — Save and Verify the Positive 5-Core Dataset

The corrected positive 5-core interactions are saved as the master
personalisation dataset. Each row represents one unique positive
user–business preference.

In [106]:
final_positive_5core = (
    positive_reviews_5core[
        [
            "review_id",
            "user_id",
            "business_id",
            "stars",
            "date",
            "interaction_sentiment"
        ]
    ]
    .sort_values(
        ["user_id", "date", "business_id", "review_id"]
    )
    .reset_index(drop=True)
)

final_positive_5core.head()

,review_id,user_id,business_id,stars,date,interaction_sentiment
0,99mk9SrwAY13O6ECUMCGMA,--T_QxqWcEu76n1daMmlLQ,DcBLYSvOuWcNReolRVr12A,5,2019-03-21 13:17:26,positive
1,GJgKnGbPBtA3IP9agv35Qw,--T_QxqWcEu76n1daMmlLQ,-HUDQ5eek6Edz3zuNrE4jw,5,2019-03-21 13:19:54,positive
2,f3zVgi6DavregKQPshqbYA,--T_QxqWcEu76n1daMmlLQ,_C7QiQQc47AOEv4PE3Kong,5,2019-03-21 14:49:29,positive
3,WawtMe7WgKEIyCiQlzcIsg,--T_QxqWcEu76n1daMmlLQ,W0xL1fk3WJqtt90mgP-7WA,5,2019-03-21 14:55:23,positive
4,Urfw2kmghVKJKh6D3O06lg,--T_QxqWcEu76n1daMmlLQ,oBNrLz4EDhiscSlbOl8uAw,5,2019-03-21 15:01:36,positive


In [107]:
final_positive_path = (
    output_dir
    / "new_orleans_positive_interactions_5core.parquet"
)

final_positive_5core.to_parquet(
    final_positive_path,
    index=False,
    engine="pyarrow"
)

print(
    f"Saved {len(final_positive_5core):,} interactions to:\n"
    f"{final_positive_path.resolve()}"
)

Saved 152,215 interactions to:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/new_orleans_positive_interactions_5core.parquet


In [108]:
saved_positive_5core = pd.read_parquet(
    final_positive_path
)

saved_positive_5core["date"] = pd.to_datetime(
    saved_positive_5core["date"],
    errors="coerce"
)

saved_user_counts = (
    saved_positive_5core
    .groupby("user_id")["business_id"]
    .nunique()
)

saved_business_counts = (
    saved_positive_5core
    .groupby("business_id")["user_id"]
    .nunique()
)

saved_positive_summary = pd.Series({
    "Saved rows":
        len(saved_positive_5core),

    "Unique users":
        saved_positive_5core["user_id"].nunique(),

    "Unique businesses":
        saved_positive_5core["business_id"].nunique(),

    "Duplicate review IDs":
        saved_positive_5core["review_id"]
        .duplicated()
        .sum(),

    "Duplicate user-business pairs":
        saved_positive_5core
        .duplicated(
            subset=["user_id", "business_id"]
        )
        .sum(),

    "Missing dates":
        saved_positive_5core["date"]
        .isna()
        .sum(),

    "Ratings below 4 stars":
        (
            saved_positive_5core["stars"] < 4
        ).sum(),

    "Minimum businesses per user":
        saved_user_counts.min(),

    "Minimum users per business":
        saved_business_counts.min(),

    "Matches in-memory row count":
        len(saved_positive_5core)
        == len(final_positive_5core),

    "File exists":
        final_positive_path.exists()
})

saved_positive_summary

Saved rows                       152215
Unique users                      14991
Unique businesses                  2516
Duplicate review IDs                  0
Duplicate user-business pairs         0
Missing dates                         0
Ratings below 4 stars                 0
Minimum businesses per user           5
Minimum users per business            5
Matches in-memory row count        True
File exists                        True
dtype: object

## Stage C1 — Create the Temporal Recommendation Split

For every retained user, the most recent positive interaction is assigned to
the test set, the second-most recent interaction is assigned to validation,
and all earlier positive interactions are used for training.

In [109]:
final_positive_path = (
    output_dir
    / "new_orleans_positive_interactions_5core.parquet"
)

positive_interactions_master = pd.read_parquet(
    final_positive_path
)

positive_interactions_master["date"] = pd.to_datetime(
    positive_interactions_master["date"],
    errors="coerce"
)

positive_interactions_master = (
    positive_interactions_master
    .sort_values(
        [
            "user_id",
            "date",
            "business_id",
            "review_id"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Master positive interactions:",
    len(positive_interactions_master)
)
print(
    "Users:",
    positive_interactions_master["user_id"].nunique()
)
print(
    "Businesses:",
    positive_interactions_master["business_id"].nunique()
)

Master positive interactions: 152215
Users: 14991
Businesses: 2516


In [110]:
positive_interactions_master["position_from_end"] = (
    positive_interactions_master
    .groupby("user_id")
    .cumcount(ascending=False)
)

positive_test = (
    positive_interactions_master.loc[
        positive_interactions_master["position_from_end"] == 0
    ]
    .drop(columns="position_from_end")
    .reset_index(drop=True)
)

positive_validation = (
    positive_interactions_master.loc[
        positive_interactions_master["position_from_end"] == 1
    ]
    .drop(columns="position_from_end")
    .reset_index(drop=True)
)

positive_train = (
    positive_interactions_master.loc[
        positive_interactions_master["position_from_end"] >= 2
    ]
    .drop(columns="position_from_end")
    .reset_index(drop=True)
)

In [111]:
split_summary = pd.DataFrame({
    "split": [
        "Training",
        "Validation",
        "Test"
    ],
    "interactions": [
        len(positive_train),
        len(positive_validation),
        len(positive_test)
    ],
    "unique_users": [
        positive_train["user_id"].nunique(),
        positive_validation["user_id"].nunique(),
        positive_test["user_id"].nunique()
    ],
    "unique_businesses": [
        positive_train["business_id"].nunique(),
        positive_validation["business_id"].nunique(),
        positive_test["business_id"].nunique()
    ],
    "earliest_date": [
        positive_train["date"].min(),
        positive_validation["date"].min(),
        positive_test["date"].min()
    ],
    "latest_date": [
        positive_train["date"].max(),
        positive_validation["date"].max(),
        positive_test["date"].max()
    ]
})

split_summary

,split,interactions,unique_users,unique_businesses,earliest_date,latest_date
0,Training,122233,14991,2516,2005-04-11 00:41:51,2022-01-19 16:36:25
1,Validation,14991,14991,1966,2005-05-18 21:15:25,2022-01-19 19:25:48
2,Test,14991,14991,1966,2005-05-18 21:38:17,2022-01-19 19:42:22


In [112]:
train_pairs = set(
    zip(
        positive_train["user_id"],
        positive_train["business_id"]
    )
)

validation_pairs = set(
    zip(
        positive_validation["user_id"],
        positive_validation["business_id"]
    )
)

test_pairs = set(
    zip(
        positive_test["user_id"],
        positive_test["business_id"]
    )
)

train_business_ids = set(
    positive_train["business_id"]
)

validation_business_ids = set(
    positive_validation["business_id"]
)

test_business_ids = set(
    positive_test["business_id"]
)

train_counts_per_user = (
    positive_train
    .groupby("user_id")
    .size()
)

validation_dates = (
    positive_validation
    .set_index("user_id")["date"]
)

test_dates = (
    positive_test
    .set_index("user_id")["date"]
)

latest_train_dates = (
    positive_train
    .groupby("user_id")["date"]
    .max()
)

split_integrity_check = pd.Series({
    "Total interactions across splits":
        (
            len(positive_train)
            + len(positive_validation)
            + len(positive_test)
        ),

    "Matches master interaction count":
        (
            len(positive_train)
            + len(positive_validation)
            + len(positive_test)
            == len(positive_interactions_master)
        ),

    "Training users":
        positive_train["user_id"].nunique(),

    "Validation users":
        positive_validation["user_id"].nunique(),

    "Test users":
        positive_test["user_id"].nunique(),

    "Minimum training interactions per user":
        train_counts_per_user.min(),

    "Train-validation pair overlap":
        len(train_pairs & validation_pairs),

    "Train-test pair overlap":
        len(train_pairs & test_pairs),

    "Validation-test pair overlap":
        len(validation_pairs & test_pairs),

    "Validation businesses absent from training":
        len(validation_business_ids - train_business_ids),

    "Test businesses absent from training":
        len(test_business_ids - train_business_ids),

    "Users whose validation predates latest training":
        (
            validation_dates
            < latest_train_dates
        ).sum(),

    "Users whose test predates validation":
        (
            test_dates
            < validation_dates
        ).sum()
})

split_integrity_check

Total interactions across splits                   152215
Matches master interaction count                     True
Training users                                      14991
Validation users                                    14991
Test users                                          14991
Minimum training interactions per user                  3
Train-validation pair overlap                           0
Train-test pair overlap                                 0
Validation-test pair overlap                            0
Validation businesses absent from training              0
Test businesses absent from training                    0
Users whose validation predates latest training         0
Users whose test predates validation                    0
dtype: object

## Stage C2 — Save and Verify the Temporal Splits

The validated positive-preference training, validation and test splits are
saved as separate Parquet files for recommender training and evaluation.

In [113]:
train_path = (
    output_dir
    / "new_orleans_positive_train.parquet"
)

validation_path = (
    output_dir
    / "new_orleans_positive_validation.parquet"
)

test_path = (
    output_dir
    / "new_orleans_positive_test.parquet"
)

positive_train.to_parquet(
    train_path,
    index=False,
    engine="pyarrow"
)

positive_validation.to_parquet(
    validation_path,
    index=False,
    engine="pyarrow"
)

positive_test.to_parquet(
    test_path,
    index=False,
    engine="pyarrow"
)

print("Saved training split:", train_path.resolve())
print("Saved validation split:", validation_path.resolve())
print("Saved test split:", test_path.resolve())

Saved training split: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/new_orleans_positive_train.parquet
Saved validation split: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/new_orleans_positive_validation.parquet
Saved test split: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/new_orleans_positive_test.parquet


In [114]:
saved_train = pd.read_parquet(train_path)
saved_validation = pd.read_parquet(validation_path)
saved_test = pd.read_parquet(test_path)

for dataframe in [
    saved_train,
    saved_validation,
    saved_test
]:
    dataframe["date"] = pd.to_datetime(
        dataframe["date"],
        errors="coerce"
    )

In [115]:
saved_split_summary = pd.DataFrame({
    "split": [
        "Training",
        "Validation",
        "Test"
    ],
    "rows": [
        len(saved_train),
        len(saved_validation),
        len(saved_test)
    ],
    "unique_users": [
        saved_train["user_id"].nunique(),
        saved_validation["user_id"].nunique(),
        saved_test["user_id"].nunique()
    ],
    "unique_businesses": [
        saved_train["business_id"].nunique(),
        saved_validation["business_id"].nunique(),
        saved_test["business_id"].nunique()
    ],
    "duplicate_review_ids": [
        saved_train["review_id"].duplicated().sum(),
        saved_validation["review_id"].duplicated().sum(),
        saved_test["review_id"].duplicated().sum()
    ],
    "duplicate_user_business_pairs": [
        saved_train.duplicated(
            subset=["user_id", "business_id"]
        ).sum(),
        saved_validation.duplicated(
            subset=["user_id", "business_id"]
        ).sum(),
        saved_test.duplicated(
            subset=["user_id", "business_id"]
        ).sum()
    ],
    "missing_dates": [
        saved_train["date"].isna().sum(),
        saved_validation["date"].isna().sum(),
        saved_test["date"].isna().sum()
    ],
    "ratings_below_4": [
        (saved_train["stars"] < 4).sum(),
        (saved_validation["stars"] < 4).sum(),
        (saved_test["stars"] < 4).sum()
    ]
})

saved_split_summary

,split,rows,unique_users,unique_businesses,duplicate_review_ids,duplicate_user_business_pairs,missing_dates,ratings_below_4
0,Training,122233,14991,2516,0,0,0,0
1,Validation,14991,14991,1966,0,0,0,0
2,Test,14991,14991,1966,0,0,0,0


In [116]:
saved_train_pairs = set(
    zip(
        saved_train["user_id"],
        saved_train["business_id"]
    )
)

saved_validation_pairs = set(
    zip(
        saved_validation["user_id"],
        saved_validation["business_id"]
    )
)

saved_test_pairs = set(
    zip(
        saved_test["user_id"],
        saved_test["business_id"]
    )
)

saved_split_integrity = pd.Series({
    "Total saved interactions":
        (
            len(saved_train)
            + len(saved_validation)
            + len(saved_test)
        ),

    "Matches master interaction count":
        (
            len(saved_train)
            + len(saved_validation)
            + len(saved_test)
            == len(positive_interactions_master)
        ),

    "Train-validation pair overlap":
        len(
            saved_train_pairs
            & saved_validation_pairs
        ),

    "Train-test pair overlap":
        len(
            saved_train_pairs
            & saved_test_pairs
        ),

    "Validation-test pair overlap":
        len(
            saved_validation_pairs
            & saved_test_pairs
        ),

    "Training file exists":
        train_path.exists(),

    "Validation file exists":
        validation_path.exists(),

    "Test file exists":
        test_path.exists()
})

saved_split_integrity

Total saved interactions            152215
Matches master interaction count      True
Train-validation pair overlap            0
Train-test pair overlap                  0
Validation-test pair overlap             0
Training file exists                  True
Validation file exists                True
Test file exists                      True
dtype: object

## Stage C3 — Create Personalisation User and Business Subsets

The user and business metadata tables are restricted to the entities in the
positive 5-core interaction dataset. These subsets are used only for the
personalisation experiment. The full discovery catalogue remains unchanged.

In [117]:
personalisation_user_ids = set(
    positive_interactions_master["user_id"]
)

personalisation_business_ids = set(
    positive_interactions_master["business_id"]
)

print(
    "Required personalisation users:",
    len(personalisation_user_ids)
)

print(
    "Required personalisation businesses:",
    len(personalisation_business_ids)
)

Required personalisation users: 14991
Required personalisation businesses: 2516


In [118]:
available_user_ids = set(
    selected_users["user_id"]
)

available_business_ids = set(
    selected_businesses["business_id"]
)

missing_personalisation_users = (
    personalisation_user_ids
    - available_user_ids
)

missing_personalisation_businesses = (
    personalisation_business_ids
    - available_business_ids
)

metadata_availability_check = pd.Series({
    "Required personalisation users":
        len(personalisation_user_ids),

    "Available user metadata rows":
        len(selected_users),

    "Required users missing from metadata":
        len(missing_personalisation_users),

    "Required personalisation businesses":
        len(personalisation_business_ids),

    "Full catalogue business rows":
        len(selected_businesses),

    "Required businesses missing from metadata":
        len(missing_personalisation_businesses)
})

metadata_availability_check

Required personalisation users               14991
Available user metadata rows                 23338
Required users missing from metadata             0
Required personalisation businesses           2516
Full catalogue business rows                  3505
Required businesses missing from metadata        0
dtype: int64

In [119]:
personalisation_users = (
    selected_users.loc[
        selected_users["user_id"].isin(
            personalisation_user_ids
        )
    ]
    .drop_duplicates(
        subset="user_id",
        keep="first"
    )
    .sort_values("user_id")
    .reset_index(drop=True)
)

personalisation_businesses = (
    selected_businesses.loc[
        selected_businesses["business_id"].isin(
            personalisation_business_ids
        )
    ]
    .drop_duplicates(
        subset="business_id",
        keep="first"
    )
    .sort_values("business_id")
    .reset_index(drop=True)
)

print(
    "Personalisation users:",
    len(personalisation_users)
)

print(
    "Personalisation businesses:",
    len(personalisation_businesses)
)

Personalisation users: 14991
Personalisation businesses: 2516


In [120]:
personalisation_metadata_check = pd.Series({
    "Personalisation user rows":
        len(personalisation_users),

    "Unique personalisation users":
        personalisation_users["user_id"].nunique(),

    "Duplicate personalisation user IDs":
        personalisation_users["user_id"]
        .duplicated()
        .sum(),

    "User IDs exactly match positive 5-core":
        set(personalisation_users["user_id"])
        == personalisation_user_ids,

    "Personalisation business rows":
        len(personalisation_businesses),

    "Unique personalisation businesses":
        personalisation_businesses[
            "business_id"
        ].nunique(),

    "Duplicate personalisation business IDs":
        personalisation_businesses[
            "business_id"
        ].duplicated().sum(),

    "Business IDs exactly match positive 5-core":
        set(personalisation_businesses["business_id"])
        == personalisation_business_ids,

    "Training users missing from subset":
        len(
            set(saved_train["user_id"])
            - set(personalisation_users["user_id"])
        ),

    "Validation users missing from subset":
        len(
            set(saved_validation["user_id"])
            - set(personalisation_users["user_id"])
        ),

    "Test users missing from subset":
        len(
            set(saved_test["user_id"])
            - set(personalisation_users["user_id"])
        ),

    "Training businesses missing from subset":
        len(
            set(saved_train["business_id"])
            - set(personalisation_businesses["business_id"])
        ),

    "Validation businesses missing from subset":
        len(
            set(saved_validation["business_id"])
            - set(personalisation_businesses["business_id"])
        ),

    "Test businesses missing from subset":
        len(
            set(saved_test["business_id"])
            - set(personalisation_businesses["business_id"])
        )
})

personalisation_metadata_check

Personalisation user rows                     14991
Unique personalisation users                  14991
Duplicate personalisation user IDs                0
User IDs exactly match positive 5-core         True
Personalisation business rows                  2516
Unique personalisation businesses              2516
Duplicate personalisation business IDs            0
Business IDs exactly match positive 5-core     True
Training users missing from subset                0
Validation users missing from subset              0
Test users missing from subset                    0
Training businesses missing from subset           0
Validation businesses missing from subset         0
Test businesses missing from subset               0
dtype: object

In [121]:
personalisation_users_path = (
    output_dir
    / "new_orleans_personalisation_users.parquet"
)

personalisation_businesses_path = (
    output_dir
    / "new_orleans_personalisation_businesses.parquet"
)

personalisation_users.to_parquet(
    personalisation_users_path,
    index=False,
    engine="pyarrow"
)

personalisation_businesses.to_parquet(
    personalisation_businesses_path,
    index=False,
    engine="pyarrow"
)

print(
    "Saved personalisation users:",
    personalisation_users_path.resolve()
)

print(
    "Saved personalisation businesses:",
    personalisation_businesses_path.resolve()
)

Saved personalisation users: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/new_orleans_personalisation_users.parquet
Saved personalisation businesses: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/new_orleans_personalisation_businesses.parquet


In [122]:
saved_personalisation_users = pd.read_parquet(
    personalisation_users_path
)

saved_personalisation_businesses = pd.read_parquet(
    personalisation_businesses_path
)

saved_personalisation_verification = pd.Series({
    "Saved user rows":
        len(saved_personalisation_users),

    "Saved users match required IDs":
        set(saved_personalisation_users["user_id"])
        == personalisation_user_ids,

    "Saved business rows":
        len(saved_personalisation_businesses),

    "Saved businesses match required IDs":
        set(
            saved_personalisation_businesses[
                "business_id"
            ]
        )
        == personalisation_business_ids,

    "Saved user file exists":
        personalisation_users_path.exists(),

    "Saved business file exists":
        personalisation_businesses_path.exists()
})

saved_personalisation_verification

Saved user rows                        14991
Saved users match required IDs          True
Saved business rows                     2516
Saved businesses match required IDs     True
Saved user file exists                  True
Saved business file exists              True
dtype: object

## Stage C4 — Final Dataset Registry and Data Freeze

The validated datasets for general discovery and personalisation are documented
in a final registry. Intermediate datasets are archived so that subsequent
notebooks use only the approved final files.

In [123]:
from datetime import datetime
from pathlib import Path
import shutil

archive_dir = (
    output_dir / "_archive_before_final_preparation"
)

archive_dir.mkdir(
    parents=True,
    exist_ok=True
)

intermediate_user_path = (
    output_dir
    / "new_orleans_users_5core.parquet"
)

if intermediate_user_path.exists():
    destination_path = (
        archive_dir
        / intermediate_user_path.name
    )

    # Preserve an existing archived copy
    if destination_path.exists():
        timestamp = datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )

        destination_path = (
            archive_dir
            / (
                f"{intermediate_user_path.stem}_"
                f"from_active_{timestamp}"
                f"{intermediate_user_path.suffix}"
            )
        )

    shutil.move(
        str(intermediate_user_path),
        str(destination_path)
    )

    print(
        "Archived intermediate user file:",
        destination_path.name
    )
else:
    print(
        "Intermediate user file is already archived."
    )

Archived intermediate user file: new_orleans_users_5core.parquet


In [124]:
dataset_specs = [
    {
        "dataset": "Full business catalogue",
        "scope": "General discovery",
        "filename": (
            "new_orleans_food_hospitality_"
            "businesses.parquet"
        ),
        "purpose": (
            "Complete New Orleans food, drink, nightlife "
            "and hospitality business catalogue."
        ),
        "implication": (
            "Preserves businesses with limited interaction "
            "or image evidence for search and cold-start analysis."
        ),
        "project_use": (
            "Business metadata embeddings, intent-driven "
            "search, discovery and cold-start evaluation."
        ),
        "status": "Final active dataset"
    },
    {
        "dataset": "Full photo metadata",
        "scope": "General discovery",
        "filename": (
            "new_orleans_food_hospitality_"
            "photos.parquet"
        ),
        "purpose": (
            "All photo records linked to businesses in "
            "the full catalogue."
        ),
        "implication": (
            "Not every business has photographs, so images "
            "must remain an optional modality."
        ),
        "project_use": (
            "Image manifest creation, image embeddings, "
            "visual discovery and visual explanation evidence."
        ),
        "status": "Final active dataset"
    },
    {
        "dataset": "Full raw review history",
        "scope": "General discovery",
        "filename": (
            "new_orleans_food_hospitality_"
            "reviews_raw.parquet"
        ),
        "purpose": (
            "Complete review text, ratings and dates linked "
            "to the full business catalogue."
        ),
        "implication": (
            "Contains mixed ratings and repeated historical "
            "user-business reviews by design. It is not used "
            "directly as positive preference data."
        ),
        "project_use": (
            "Review-text preparation, business understanding, "
            "sentiment evidence and explanation retrieval."
        ),
        "status": "Final active dataset"
    },
    {
        "dataset": "Location comparison",
        "scope": "Dataset selection",
        "filename": (
            "top_five_location_image_comparison.csv"
        ),
        "purpose": (
            "Comparison of the leading candidate cities "
            "before selecting New Orleans."
        ),
        "implication": (
            "Provides documented evidence for the geographic "
            "selection decision."
        ),
        "project_use": (
            "Methodology, dataset-selection justification "
            "and descriptive analysis."
        ),
        "status": "Final documentation dataset"
    },
    {
        "dataset": "Personalisation positive 5-core",
        "scope": "Personalisation",
        "filename": (
            "new_orleans_positive_"
            "interactions_5core.parquet"
        ),
        "purpose": (
            "Unique latest four- and five-star interactions "
            "after iterative 5-core filtering."
        ),
        "implication": (
            "Supports users with sufficient preference history, "
            "but findings apply to relatively active users and "
            "well-connected businesses."
        ),
        "project_use": (
            "Master personalisation interaction dataset, "
            "collaborative filtering and KGRec graph edges."
        ),
        "status": "Final active dataset"
    },
    {
        "dataset": "Personalisation training interactions",
        "scope": "Personalisation",
        "filename": (
            "new_orleans_positive_train.parquet"
        ),
        "purpose": (
            "Earlier positive interactions for each retained user."
        ),
        "implication": (
            "Contains at least three training preferences per user."
        ),
        "project_use": (
            "Collaborative filtering, KGRec training and "
            "user-preference learning."
        ),
        "status": "Final active dataset"
    },
    {
        "dataset": "Personalisation validation interactions",
        "scope": "Personalisation",
        "filename": (
            "new_orleans_positive_validation.parquet"
        ),
        "purpose": (
            "Second-most-recent positive interaction for "
            "each retained user."
        ),
        "implication": (
            "Must be used for model selection rather than "
            "final performance reporting."
        ),
        "project_use": (
            "Hyperparameter selection, early stopping and "
            "model comparison."
        ),
        "status": "Final active dataset"
    },
    {
        "dataset": "Personalisation test interactions",
        "scope": "Personalisation",
        "filename": (
            "new_orleans_positive_test.parquet"
        ),
        "purpose": (
            "Most recent positive interaction for each "
            "retained user."
        ),
        "implication": (
            "Must remain untouched during model development "
            "to preserve unbiased final evaluation."
        ),
        "project_use": (
            "Final Recall@K, NDCG@K, MAP@K and other "
            "personalised-ranking evaluation."
        ),
        "status": "Final active dataset"
    },
    {
        "dataset": "Personalisation users",
        "scope": "Personalisation",
        "filename": (
            "new_orleans_personalisation_users.parquet"
        ),
        "purpose": (
            "User metadata for the users retained in the "
            "positive 5-core."
        ),
        "implication": (
            "Represents only the personalisation experiment "
            "and not all users in the raw review history."
        ),
        "project_use": (
            "Personalisation user nodes, descriptive analysis "
            "and supplementary user features."
        ),
        "status": "Final active dataset"
    },
    {
        "dataset": "Personalisation businesses",
        "scope": "Personalisation",
        "filename": (
            "new_orleans_personalisation_businesses.parquet"
        ),
        "purpose": (
            "Business metadata for businesses retained in "
            "the positive 5-core."
        ),
        "implication": (
            "This is a subset of the full 3,505-business "
            "discovery catalogue."
        ),
        "project_use": (
            "Personalisation business nodes, KGRec and "
            "collaborative-ranking experiments."
        ),
        "status": "Final active dataset"
    }
]

In [125]:
import pandas as pd

registry_rows = []

for specification in dataset_specs:
    file_path = (
        output_dir
        / specification["filename"]
    )

    if not file_path.exists():
        raise FileNotFoundError(
            f"Required final dataset is missing: "
            f"{file_path}"
        )

    if file_path.suffix == ".parquet":
        dataframe = pd.read_parquet(file_path)

    elif file_path.suffix == ".csv":
        dataframe = pd.read_csv(file_path)

    else:
        raise ValueError(
            f"Unsupported file type: {file_path.suffix}"
        )

    registry_rows.append({
        "dataset": specification["dataset"],
        "scope": specification["scope"],
        "filename": specification["filename"],
        "rows": len(dataframe),
        "columns": len(dataframe.columns),

        "unique_users": (
            dataframe["user_id"].nunique()
            if "user_id" in dataframe.columns
            else None
        ),

        "unique_businesses": (
            dataframe["business_id"].nunique()
            if "business_id" in dataframe.columns
            else None
        ),

        "earliest_date": (
            pd.to_datetime(
                dataframe["date"],
                errors="coerce"
            ).min()
            if "date" in dataframe.columns
            else None
        ),

        "latest_date": (
            pd.to_datetime(
                dataframe["date"],
                errors="coerce"
            ).max()
            if "date" in dataframe.columns
            else None
        ),

        "file_size_mb": round(
            file_path.stat().st_size
            / (1024 ** 2),
            2
        ),

        "purpose": specification["purpose"],
        "implication": specification["implication"],
        "project_use": specification["project_use"],
        "status": specification["status"]
    })

dataset_registry = pd.DataFrame(
    registry_rows
)

dataset_registry

,dataset,scope,filename,rows,columns,unique_users,unique_businesses,earliest_date,latest_date,file_size_mb,purpose,implication,project_use,status
0,Full business catalogue,General discovery,new_orleans_food_hospitality_businesses.parquet,3505,20,NaN,3505.0,NaT,NaT,0.82,"Complete New Orleans food, drink, nightlife an...",Preserves businesses with limited interaction ...,"Business metadata embeddings, intent-driven se...",Final active dataset
1,Full photo metadata,General discovery,new_orleans_food_hospitality_photos.parquet,19456,6,NaN,1977.0,NaT,NaT,1.05,All photo records linked to businesses in the ...,"Not every business has photographs, so images ...","Image manifest creation, image embeddings, vis...",Final active dataset
2,Full raw review history,General discovery,new_orleans_food_hospitality_reviews_raw.parquet,559117,13,222493.0,3505.0,2005-03-14 18:07:51,2022-01-19 19:47:59,212.57,"Complete review text, ratings and dates linked...",Contains mixed ratings and repeated historical...,"Review-text preparation, business understandin...",Final active dataset
3,Location comparison,Dataset selection,top_five_location_image_comparison.csv,5,14,NaN,NaN,NaT,NaT,0.00,Comparison of the leading candidate cities bef...,Provides documented evidence for the geographi...,"Methodology, dataset-selection justification a...",Final documentation dataset
4,Personalisation positive 5-core,Personalisation,new_orleans_positive_interactions_5core.parquet,152215,6,14991.0,2516.0,2005-04-11 00:41:51,2022-01-19 19:42:22,5.69,Unique latest four- and five-star interactions...,Supports users with sufficient preference hist...,"Master personalisation interaction dataset, co...",Final active dataset
5,Personalisation training interactions,Personalisation,new_orleans_positive_train.parquet,122233,6,14991.0,2516.0,2005-04-11 00:41:51,2022-01-19 16:36:25,4.70,Earlier positive interactions for each retaine...,Contains at least three training preferences p...,"Collaborative filtering, KGRec training and us...",Final active dataset
6,Personalisation validation interactions,Personalisation,new_orleans_positive_validation.parquet,14991,6,14991.0,1966.0,2005-05-18 21:15:25,2022-01-19 19:25:48,0.93,Second-most-recent positive interaction for ea...,Must be used for model selection rather than f...,"Hyperparameter selection, early stopping and m...",Final active dataset
7,Personalisation test interactions,Personalisation,new_orleans_positive_test.parquet,14991,6,14991.0,1966.0,2005-05-18 21:38:17,2022-01-19 19:42:22,0.93,Most recent positive interaction for each reta...,Must remain untouched during model development...,"Final Recall@K, NDCG@K, MAP@K and other person...",Final active dataset
8,Personalisation users,Personalisation,new_orleans_personalisation_users.parquet,14991,9,14991.0,NaN,NaT,NaT,0.66,User metadata for the users retained in the po...,Represents only the personalisation experiment...,"Personalisation user nodes, descriptive analys...",Final active dataset
9,Personalisation businesses,Personalisation,new_orleans_personalisation_businesses.parquet,2516,20,NaN,2516.0,NaT,NaT,0.65,Business metadata for businesses retained in t...,"This is a subset of the full 3,505-business di...","Personalisation business nodes, KGRec and coll...",Final active dataset


In [126]:
registry_path = (
    output_dir
    / "dataset_registry.csv"
)

dataset_registry.to_csv(
    registry_path,
    index=False,
    encoding="utf-8"
)

saved_dataset_registry = pd.read_csv(
    registry_path
)

print(
    "Registry saved to:",
    registry_path.resolve()
)

print(
    "Registry entries:",
    len(saved_dataset_registry)
)

Registry saved to: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset/dataset_registry.csv
Registry entries: 10


In [127]:
expected_active_files = {
    specification["filename"]
    for specification in dataset_specs
}

expected_active_files.add(
    "dataset_registry.csv"
)

actual_active_files = {
    file_path.name
    for file_path in output_dir.iterdir()
    if file_path.is_file()
}

missing_final_files = (
    expected_active_files
    - actual_active_files
)

unexpected_active_files = (
    actual_active_files
    - expected_active_files
)

data_freeze_check = pd.Series({
    "Expected active files":
        len(expected_active_files),

    "Actual active files":
        len(actual_active_files),

    "Missing final files":
        len(missing_final_files),

    "Unexpected active files":
        len(unexpected_active_files),

    "Registry file exists":
        registry_path.exists(),

    "Registry entries":
        len(saved_dataset_registry),

    "General discovery datasets":
        (
            saved_dataset_registry["scope"]
            == "General discovery"
        ).sum(),

    "Personalisation datasets":
        (
            saved_dataset_registry["scope"]
            == "Personalisation"
        ).sum(),

    "Dataset preparation frozen":
        (
            len(missing_final_files) == 0
            and len(unexpected_active_files) == 0
            and registry_path.exists()
        )
})

data_freeze_check

Expected active files           11
Actual active files             11
Missing final files              0
Unexpected active files          0
Registry file exists          True
Registry entries                10
General discovery datasets       3
Personalisation datasets         6
Dataset preparation frozen    True
dtype: object

In [128]:
print("Missing final files:")
for filename in sorted(missing_final_files):
    print("-", filename)

print("\nUnexpected active files:")
for filename in sorted(unexpected_active_files):
    print("-", filename)

print("\nFinal active files:")
for filename in sorted(actual_active_files):
    print("-", filename)

Missing final files:

Unexpected active files:

Final active files:
- dataset_registry.csv
- new_orleans_food_hospitality_businesses.parquet
- new_orleans_food_hospitality_photos.parquet
- new_orleans_food_hospitality_reviews_raw.parquet
- new_orleans_personalisation_businesses.parquet
- new_orleans_personalisation_users.parquet
- new_orleans_positive_interactions_5core.parquet
- new_orleans_positive_test.parquet
- new_orleans_positive_train.parquet
- new_orleans_positive_validation.parquet
- top_five_location_image_comparison.csv
